# 🌿 EcoGPT — Multi-Agent RAG Environmental Advisory System
### AMD–TCS Hackathon | Location-aware environmental intelligence from IoT sensor streams

**Pipeline:** Location → Sensor Ingestion → Multi-Agent Analysis (8 agents) → RAG-Augmented Synthesis → Structured Recommendations

**Works anywhere on the globe.** Give it any city or lat/lon. Where local sensor data exists it is used directly; elsewhere EcoGPT falls back to climate-zone modelling and marks every figure *(estimated)*.

**LLM backends (auto-detected, in priority order):**
1. **Ollama** — `mistral:7b-instruct` / `llama3.1:8b` running locally (AMD ROCm accelerated)
2. **HuggingFace Inference API** — set `HF_TOKEN` env var
3. **Deterministic engine** — every agent has a rule-based scientific core, so the full pipeline runs with *no LLM at all* (LLMs only polish the narrative)

> Run cells top-to-bottom. Cell 1 installs missing packages and reports what's available.


In [ ]:
# ============================================================
# Cell 1: Environment Setup & Imports
# Installs missing packages, detects capabilities, prints report
# ============================================================
import sys, subprocess, importlib, warnings, os, json, math, random, re, datetime
warnings.filterwarnings("ignore")

PKGS = [  # (import_name, pip_name, required?)
    ("pandas", "pandas", True), ("numpy", "numpy", True),
    ("matplotlib", "matplotlib", True), ("requests", "requests", True),
    ("folium", "folium", False), ("plotly", "plotly", False),
    ("ipywidgets", "ipywidgets", False),
    ("chromadb", "chromadb", False),
    ("sentence_transformers", "sentence-transformers", False),
    ("rank_bm25", "rank-bm25", False),
    ("geopy", "geopy", False),
    ("google.adk", "google-adk", False),
]

def _ensure(mod, pip_name):
    try:
        importlib.import_module(mod); return True
    except Exception:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name],
                                  stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            importlib.invalidate_caches()
            importlib.import_module(mod); return True
        except Exception:
            return False

CAPS = {mod: _ensure(mod, pip) for mod, pip, _ in PKGS}

import pandas as pd, numpy as np
import matplotlib.pyplot as plt

if CAPS["folium"]: import folium
if CAPS["geopy"]:
    from geopy.geocoders import Nominatim
if CAPS["rank_bm25"]:
    from rank_bm25 import BM25Okapi

print("=" * 56)
print("EcoGPT capability report")
print("=" * 56)
for mod, pip, req in PKGS:
    mark = "✅" if CAPS[mod] else ("❌ REQUIRED" if req else "⚪ optional (fallback active)")
    print(f"  {pip:<25} {mark}")

# AMD ROCm / GPU detection (informational)
GPU = False
try:
    import torch
    GPU = torch.cuda.is_available()  # True on ROCm builds too
    print(f"  torch GPU (CUDA/ROCm)     {'✅ ' + torch.cuda.get_device_name(0) if GPU else '⚪ CPU mode'}")
except Exception:
    print("  torch                     ⚪ not installed (CPU/sklearn fallbacks active)")
print("=" * 56)

RNG = np.random.default_rng(42)
random.seed(42)
DATA_DIR = "."
VECTORDB_PATH = "./ecogpt_vectordb"


In [ ]:
# ============================================================
# Cell 2: Sensor Data — load reference CSV + generate a large,
# realistic, multi-location synthetic dataset calibrated to it
# ============================================================
# The reference field CSV (~2k rows, Kolkata) is small and has gaps
# (RAWPM almost entirely missing, DD spikes, a few NYC test rows).
# For robust analytics we synthesize a 6-city, 30-day, 15-min-interval
# dataset whose Kolkata distributions are calibrated to the real CSV.
# Drop ANY real CSV with the same schema next to this notebook and it
# is automatically merged in.

SENSOR_COLS = ["MQ2","MQ7","MQ135","NO2","C2H5OH","VOC","CO","HMD","TMP","HI","RAWPM","DD"]

def load_reference_csvs(pattern_dir="."):
    """Load every enviro_sensorvalues_*.csv found beside the notebook."""
    import glob
    frames = []
    for path in sorted(glob.glob(os.path.join(pattern_dir, "enviro_sensorvalues_*.csv"))):
        try:
            df = pd.read_csv(path)
            df["time"] = pd.to_datetime(df["time"], errors="coerce")
            df["source"] = os.path.basename(path)
            frames.append(df)
            print(f"  loaded {path}: {len(df)} rows")
        except Exception as e:
            print(f"  skipped {path}: {e}")
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def heat_index_c(t_c, rh):
    """NOAA heat index (Rothfusz), input/output in °C."""
    t = t_c * 9/5 + 32
    if t < 80:
        hi = 0.5*(t + 61.0 + (t-68.0)*1.2 + rh*0.094)
    else:
        hi = (-42.379 + 2.04901523*t + 10.14333127*rh - .22475541*t*rh
              - .00683783*t*t - .05481717*rh*rh + .00122874*t*t*rh
              + .00085282*t*rh*rh - .00000199*t*t*rh*rh)
    return (hi - 32) * 5/9

# City profiles: (lat, lon, base_temp °C, base_rh %, pollution_factor, diurnal_amp)
CITY_PROFILES = {
    "Kolkata":     (22.5570,  88.4940, 32.0, 60.0, 1.00, 4.0),
    "Delhi":       (28.6139,  77.2090, 34.0, 45.0, 1.35, 7.0),
    "Mumbai":      (19.0760,  72.8777, 30.0, 74.0, 0.85, 3.0),
    "Sundarbans":  (21.9497,  88.9468, 29.0, 78.0, 0.25, 3.5),  # rural/riverine
    "London":      (51.5074,  -0.1278, 14.0, 75.0, 0.45, 4.5),
    "Nairobi":     ( -1.2921, 36.8219, 21.0, 62.0, 0.55, 6.0),
}

def generate_synthetic_dataset(days=30, freq_min=15, ref_df=None):
    """Big calibrated dataset. Kolkata channel statistics are anchored to the
    reference CSV means/stds when it is available."""
    # Calibration anchors (fallback = stats measured from the uploaded field CSV)
    anchors = {"MQ2":(4.43,1.15),"MQ7":(4.78,1.07),"MQ135":(8.59,1.58),"NO2":(5.80,1.05),
               "C2H5OH":(58.5,9.8),"VOC":(52.1,8.4),"CO":(81.5,13.3),"HMD":(59.6,9.0),
               "TMP":(32.0,1.5),"DD":(253.0,25.0),"RAWPM":(95.0,30.0)}
    if ref_df is not None and len(ref_df) > 100:
        k = ref_df[(ref_df["LAT"].sub(22.557).abs() < 0.05)]
        for c in SENSOR_COLS:
            if c in k and k[c].notna().sum() > 50:
                anchors[c] = (float(k[c].mean()), max(float(k[c].std()), 1e-3))

    end = pd.Timestamp.now().floor("h")
    times = pd.date_range(end - pd.Timedelta(days=days), end, freq=f"{freq_min}min")
    rows = []
    for city, (lat, lon, bt, brh, pf, amp) in CITY_PROFILES.items():
        n = len(times)
        hod = times.hour.values + times.minute.values/60
        # diurnal cycles: temp peaks 14:00, traffic pollution peaks 9:00 & 19:00
        temp = bt + amp*np.sin((hod-8)/24*2*np.pi) + RNG.normal(0, 0.8, n)
        rh = np.clip(brh - 1.2*(temp-bt) + RNG.normal(0, 4, n), 15, 99)
        traffic = 1 + 0.55*(np.exp(-((hod-9)**2)/6) + np.exp(-((hod-19)**2)/6))
        def chan(mu, sd, extra=1.0):
            base = mu*pf*extra*traffic if city != "Kolkata" else mu*extra*traffic/np.mean(traffic)
            return np.clip(base + RNG.normal(0, sd, n), 0.01, None)
        r = pd.DataFrame({
            "MQ2":  chan(*anchors["MQ2"]),  "MQ7":  chan(*anchors["MQ7"]),
            "MQ135":chan(*anchors["MQ135"]),"NO2":  chan(*anchors["NO2"]),
            "C2H5OH":chan(*anchors["C2H5OH"]),"VOC": chan(*anchors["VOC"]),
            "CO":   chan(*anchors["CO"]),
            "HMD": rh, "TMP": temp,
            "RAWPM": chan(*anchors["RAWPM"]),
            "DD":   chan(*anchors["DD"]),
            "LAT": lat + RNG.normal(0, 0.004, n), "LON": lon + RNG.normal(0, 0.004, n),
            "time": times,
        })
        r["HI"] = [heat_index_c(t, h) for t, h in zip(r["TMP"], r["HMD"])]
        r["source"] = f"synthetic_{city}"
        rows.append(r)
    out = pd.concat(rows, ignore_index=True)
    # inject realistic anomalies (1% pollution spike events)
    spike = RNG.random(len(out)) < 0.01
    out.loc[spike, ["CO","NO2","VOC","RAWPM","DD"]] *= RNG.uniform(1.8, 3.0)
    return out

print("Reference CSVs:")
ref_df = load_reference_csvs(DATA_DIR)
sensor_df = generate_synthetic_dataset(days=30, freq_min=15, ref_df=ref_df if len(ref_df) else None)
if len(ref_df):
    sensor_df = pd.concat([ref_df.drop(columns=["id"], errors="ignore"), sensor_df],
                          ignore_index=True)
sensor_df = sensor_df.dropna(subset=["LAT","LON","time"]).reset_index(drop=True)
sensor_df.to_csv("ecogpt_master_dataset.csv", index=False)
print(f"\nMaster dataset: {len(sensor_df):,} rows | "
      f"{sensor_df['time'].min()} → {sensor_df['time'].max()} | "
      f"{sensor_df[['LAT','LON']].round(1).drop_duplicates().shape[0]} location clusters")
sensor_df.describe().T.round(2)


In [ ]:
# ============================================================
# Cell 2b (optional): Kaggle dataset enrichment
#   Adds real city-level baselines to strengthen analysis where
#   neither IoT sensors nor live feeds are available.
#   Setup (one-time): pip install kagglehub, then place your
#   kaggle.json (kaggle.com → Account → Create API Token) at
#   ~/.kaggle/kaggle.json — or set KAGGLE_USERNAME/KAGGLE_KEY env vars.
#   Skips silently when unavailable; the pipeline works without it.
#
#   Useful datasets indexed here:
#   • hasibalmuzdadid/global-air-pollution-dataset  → AQI for ~23k cities (loaded below)
#   • Other good additions (same pattern): global temperature records,
#     world cities population, country forest-cover series.
# ============================================================
KAGGLE_AQ = None

def load_kaggle_baselines():
    global KAGGLE_AQ
    try:
        import kagglehub, glob as _gl
        path = kagglehub.dataset_download("hasibalmuzdadid/global-air-pollution-dataset")
        df = pd.read_csv(_gl.glob(os.path.join(path, "*.csv"))[0])
        df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
        df = df.dropna(subset=["city"])
        KAGGLE_AQ = df
        print(f"✅ Kaggle global air-pollution baselines: {len(df):,} cities "
              f"(columns: {', '.join(df.columns[:6])} …)")
    except Exception as e:
        print(f"⚪ Kaggle enrichment skipped ({type(e).__name__}). "
              "To enable: pip install kagglehub + kaggle.json API token. "
              "Pipeline runs fine without it (live Open-Meteo feeds cover all locations).")

def kaggle_city_baseline(city: str):
    """City-level AQI baseline from the Kaggle dataset, or None."""
    if KAGGLE_AQ is None or not city: return None
    name = city.lower().split(",")[0].strip()
    hit = KAGGLE_AQ[KAGGLE_AQ["city"].str.lower() == name]
    if not len(hit): return None
    row = hit.iloc[0]
    aqi_col = next((c for c in ("aqi_value", "aqi") if c in KAGGLE_AQ.columns), None)
    pm_col = next((c for c in KAGGLE_AQ.columns if "pm2" in c and "value" in c), None)
    return {"aqi": float(row[aqi_col]) if aqi_col else float("nan"),
            "pm25_aqi": float(row[pm_col]) if pm_col else float("nan"),
            "source": "Kaggle: global-air-pollution-dataset"}

load_kaggle_baselines()


In [ ]:
# ============================================================
# Cell 3: Exploratory Data Analysis + sensor map
# ============================================================
fig, axes = plt.subplots(3, 4, figsize=(18, 10))
for ax, col in zip(axes.flat, SENSOR_COLS):
    for src, grp in sensor_df.groupby(sensor_df["source"].str.replace("synthetic_", "")):
        if len(grp) > 200:
            grp = grp.sample(200, random_state=1)
        ax.hist(grp[col].dropna(), bins=30, alpha=0.45, label=src)
    ax.set_title(col, fontsize=10); ax.tick_params(labelsize=7)
axes.flat[0].legend(fontsize=6)
plt.suptitle("Sensor distributions by location", y=1.01)
plt.tight_layout(); plt.show()

# Diurnal pollution profile (Kolkata)
kol = sensor_df[(sensor_df["LAT"].sub(22.557).abs() < 0.05)].copy()
kol["hour"] = pd.to_datetime(kol["time"]).dt.hour
fig, ax = plt.subplots(figsize=(10, 3.5))
for c in ["CO", "NO2", "VOC"]:
    prof = kol.groupby("hour")[c].mean()
    ax.plot(prof.index, prof / prof.max(), marker="o", ms=3, label=c)
ax.set(title="Kolkata diurnal pollution profile (normalised)", xlabel="hour of day")
ax.legend(); plt.tight_layout(); plt.show()

if CAPS["folium"]:
    m = folium.Map(location=[22.557, 88.494], zoom_start=3, tiles="CartoDB positron")
    for src, grp in sensor_df.groupby("source"):
        g = grp.dropna(subset=["LAT", "LON"])
        if not len(g): continue
        lat, lon = g["LAT"].median(), g["LON"].median()
        co = g["CO"].mean()
        folium.CircleMarker(
            [lat, lon], radius=6 + min(co / 10, 12),
            color="red" if co > 60 else "orange" if co > 30 else "green",
            fill=True, tooltip=f"{src}: {len(g):,} readings | CO {co:.1f} ppm"
        ).add_to(m)
    display(m)
else:
    print("folium unavailable — skipping map")


In [ ]:
# ============================================================
# Cell 4: Knowledge Base Documents (curated, multi-climate-zone)
# Sources condensed from: WHO AQ Guidelines 2021, CPCB/NAAQS India,
# FAO urban forestry handbook, IPCC AR6 WGIII Ch.7, i-Tree species
# data, CGWB/CWC water guidelines, ICAR & FAO soil manuals,
# India Biodiversity Portal, Miyawaki method literature.
# ============================================================
KNOWLEDGE_DOCS = [
# ---------- SPECIES / PLANTATION ----------
{"id":"sp_trop_01","topic":"species","climate_zone":"tropical","source":"India Biodiversity Portal / i-Tree",
 "text":"Native tree species for tropical wet and monsoon climates (Köppen Aw/Am, e.g. Kolkata, West Bengal): "
 "Banyan (Ficus benghalensis) — keystone fig, massive canopy, sequesters ~28 kg CO2/yr mature, medium water; "
 "Neem (Azadirachta indica) — drought-hardy air purifier strong on NO2 and SO2, ~22 kg CO2/yr, low water; "
 "Arjun (Terminalia arjuna) — riparian specialist for riverbanks and pond bunds, ~25 kg CO2/yr, high water; "
 "Kadamba (Neolamarckia cadamba) — fast-growing monsoon bloomer, pollinator magnet, ~24 kg CO2/yr; "
 "Sal (Shorea robusta) — long-lived forest dominant for restoration blocks, ~26 kg CO2/yr; "
 "Mahua (Madhuca longifolia) and Palash (Butea monosperma) — dry-deciduous, livelihood and pollinator value; "
 "Krishnachura (Delonix regia) — avenue flowering tree, heat tolerant, ~18 kg CO2/yr, low water; "
 "Shimul (Bombax ceiba) — emergent layer, bird habitat; Jackfruit (Artocarpus heterophyllus) — food + shade. "
 "Plant at monsoon onset (June–July). Avoid monocultures; mix 8–12 species minimum."},
{"id":"sp_trop_02","topic":"species","climate_zone":"tropical","source":"FAO urban forestry handbook",
 "text":"Dense-urban tropical planting (>5000 persons/km²): compact flowering trees Cassia fistula (Amaltas, ~15 kg CO2/yr, low water) "
 "and Lagerstroemia speciosa (Pride of India, ~14 kg CO2/yr); air-purifying shrubs and hedges: Murraya paniculata, Hibiscus rosa-sinensis, "
 "Tabernaemontana divaricata. Vertical gardens with Epipremnum aureum, Chlorophytum comosum (VOC removal), Spathiphyllum wallisii. "
 "Rooftop greening with sedums and native grasses cuts roof surface temperature 25–40°C and building cooling load 15–30%. "
 "Miyawaki micro-forests: 3–5 saplings/m² of 15–30 native species reach canopy closure in 3 years, 30x density of conventional planting."},
{"id":"sp_arid_01","topic":"species","climate_zone":"arid","source":"FAO dryland forestry",
 "text":"Arid and semi-arid (Köppen BWh/BSh, e.g. Delhi pre-monsoon, Rajasthan, Sahel): Khejri (Prosopis cineraria — native, NOT the invasive P. juliflora), "
 "Babul (Vacachellia nilotica), Indian jujube (Ziziphus mauritiana), Neem, Pongamia pinnata (biodiesel + N-fixing, ~20 kg CO2/yr), "
 "Drumstick (Moringa oleifera, fast nutrition tree). Water budget <600 mm/yr: choose deep-rooted phreatophytes, drip-basin planting, "
 "mulch pits. Avoid Eucalyptus in groundwater-stressed zones — draws 30–90 L/day/tree."},
{"id":"sp_temp_01","topic":"species","climate_zone":"temperate","source":"i-Tree / European urban forestry",
 "text":"Temperate oceanic/continental (Köppen Cfb/Dfb, e.g. London, Berlin): English oak (Quercus robur, ~30 kg CO2/yr mature), "
 "Small-leaved lime (Tilia cordata, pollinator keystone), London plane (Platanus × acerifolia, pollution-tolerant avenue standard), "
 "Silver birch (Betula pendula, PM capture on leaf hairs), Field maple (Acer campestre), Rowan (Sorbus aucuparia, bird forage). "
 "Plant bare-root in dormancy (Nov–Mar). Street pits ≥6 m³ soil volume for 60% canopy survival at 20 years."},
{"id":"sp_savanna_01","topic":"species","climate_zone":"tropical_highland","source":"Kenya Forestry Research Institute",
 "text":"Tropical highland/savanna (e.g. Nairobi, Addis Ababa): Croton megalocarpus (~25 kg CO2/yr), Markhamia lutea, "
 "Cordia africana, Podocarpus falcatus (indigenous conifer), Acacia xanthophloea (fever tree, riparian), Prunus africana (medicinal, IUCN vulnerable — propagate). "
 "Avoid invasive: Lantana camara, Eucalyptus monocultures near wetlands."},
{"id":"sp_invasive","topic":"species","climate_zone":"global","source":"IUCN GISD / CABI",
 "text":"NEVER recommend invasive species: Prosopis juliflora (vilayati babul — allelopathic, groundwater depletion), Lantana camara (forest understory choker), "
 "Eichhornia crassipes (water hyacinth — covers ponds, kills fisheries; if present recommend manual + bio control with Neochetina weevils and conversion to compost/biogas), "
 "Parthenium hysterophorus (allergenic weed), Leucaena leucocephala (aggressive seeder in disturbed land), Acacia mearnsii (riparian invader). "
 "Polyculture rule: no species >15% of total planting; minimum 10 species per hectare for resilience."},
{"id":"sp_biofilter","topic":"species","climate_zone":"global","source":"NASA Clean Air Study / Pugh et al. 2012",
 "text":"Bio-filter species by pollutant: NO2 — Ficus benjamina, Hedera helix (ivy screens on roadside railings), Azadirachta indica; "
 "PM2.5/dust — Ficus elastica, Tillandsia usneoides (Spanish moss epiphyte), conifers and Betula (high leaf-hair capture), Neem hedges; "
 "VOC/formaldehyde — Chlorophytum comosum (spider plant), Spathiphyllum wallisii (peace lily), Epipremnum aureum; "
 "SO2 — Tamarindus indica, Mangifera indica. Green walls in street canyons cut street-level NO2 up to 40% and PM 60% (Pugh et al.); "
 "tree canopy citywide typically reduces PM2.5 2–10%. A mature urban tree intercepts 1.4 kg of PM and pollutant gases per year on average."},
# ---------- AIR POLLUTION ----------
{"id":"air_01","topic":"pollution","climate_zone":"global","source":"WHO Global Air Quality Guidelines 2021",
 "text":"WHO 2021 guideline values: PM2.5 annual 5 µg/m³, 24-h 15 µg/m³; NO2 annual 10 µg/m³, 24-h 25 µg/m³; CO 24-h 4 mg/m³. "
 "India NAAQS (CPCB): PM2.5 annual 40, 24-h 60 µg/m³; NO2 annual 40 µg/m³. AQI categories: Good 0–50, Moderate 51–100, "
 "Unhealthy for Sensitive Groups 101–150, Unhealthy 151–200, Very Unhealthy 201–300, Hazardous >300. "
 "Health burden: each 10 µg/m³ PM2.5 above guideline ≈ +6% cardiopulmonary mortality risk."},
{"id":"air_02","topic":"pollution","climate_zone":"global","source":"CPCB GRAP / source apportionment studies",
 "text":"Pollution source signatures: traffic — elevated NO2 + CO with morning/evening peaks; industrial — sustained VOC + MQ135 (NH3/NOx) with weekday pattern; "
 "biomass/refuse burning — MQ2 smoke + CO spikes evening/winter; construction — dust density (DD) and coarse PM spikes daytime. "
 "Short-term (24–72 h) actions by AQI: >200 — halt construction, odd-even traffic, water-sprinkle roads, N95 advisories, close schools outdoor activity; "
 "151–200 — restrict diesel gensets, intensify mechanised road sweeping, public transport fare incentives. "
 "Medium-term (1–6 mo): green barriers along arterials (3-row Neem/Ficus hedges), anti-smog guns at hotspots, LPG conversion of street food stalls, filtered ventilation in schools/hospitals. "
 "Long-term (1–5 yr): urban forestry corridors, EV transition zones with charging mandates, industrial relocation/buffer greenbelts (500 m), green building codes, district cooling."},
{"id":"air_03","topic":"pollution","climate_zone":"global","source":"EPA / UHI literature",
 "text":"Urban heat island: flag when heat index exceeds ambient temperature by >3°C or night-time urban-rural delta >2°C. "
 "Mitigation: cool roofs (albedo >0.65 cuts roof temp ~28°C), 30% tree canopy target lowers ambient 1–3°C and peak surface 11°C, "
 "permeable pavements with evapotranspiration, blue infrastructure (ponds reduce local temp 1–2°C downwind 100–300 m). "
 "Heat-health: HI >41°C 'danger' — heat cramps likely; >54°C 'extreme danger' — heat stroke imminent. Open cooling centres, shift outdoor labour hours."},
# ---------- WATER ----------
{"id":"wat_01","topic":"water","climate_zone":"global","source":"CGWB Master Plan / CWC guidelines",
 "text":"Rainwater harvesting: harvestable volume (L/yr) = roof area m² × annual rainfall mm × 0.8 runoff coefficient. "
 "Kolkata rainfall ~1,800 mm/yr → a 100 m² roof yields ~144,000 L/yr, meeting ~40% of a 5-person household demand (135 LPCD norm). "
 "Recharge structures: percolation pits 1–2 m³ per 100 m² roof, recharge trenches along boundaries, defunct borewell recharge shafts. "
 "Urban mandate benchmark: structures compulsory on plots >300 m² in most Indian municipal bylaws. "
 "Groundwater table response: dense RWH retrofits typically raise local water table 0.3–1.0 m within 3–5 monsoons (CGWB pilot data)."},
{"id":"wat_02","topic":"water","climate_zone":"global","source":"Central Water Commission / wetland restoration manuals",
 "text":"Water body (pond/lake/wetland) restoration sequence: 1) catchment survey + sewage interception (divert or treat inflows first — restoration fails otherwise); "
 "2) desilting in dry season to original bed level, reuse silt on bunds/agriculture after testing; 3) bund strengthening with Vetiver grass (Chrysopogon zizanioides) hedgerows — roots to 3 m, halve embankment erosion; "
 "4) riparian buffer 10–30 m: Arjun, Jamun (Syzygium cumini), Bamboo (Bambusa balcooa), Typha and Phragmites reed beds as final-polish bioremediation; "
 "5) aquatic vegetation: Nelumbo nucifera (lotus) and Nymphaea water lilies ≤30% surface cover; remove Eichhornia fully; "
 "6) constructed wetland / bioswale interception of stormwater first-flush; 7) native fish restocking (Rohu, Catla, Mrigal in Gangetic plains) after DO >4 mg/L. "
 "East Kolkata Wetlands model: sewage-fed aquaculture treats ~750 MLD naturally — protect peri-urban wetlands as treatment + livelihood infrastructure."},
{"id":"wat_03","topic":"water","climate_zone":"global","source":"FAO irrigation efficiency",
 "text":"Irrigation efficiency: flood ~40% efficient, sprinkler ~70%, drip 90%+. Converting 1 ha paddy-adjacent vegetable cultivation from flood to drip saves "
 "~3–4 million L/yr. Soil-moisture-sensor scheduling saves further 15–25%. Water stress classification by ambient signals: "
 "RH <40% + T >35°C = high evaporative stress (pan evaporation >8 mm/day); RH 40–70% moderate; RH >70% humid (fungal risk, drainage priority). "
 "Check dams on first/second-order streams: 0.5–2 m height, recharge 5,000–20,000 m³/structure/yr in suitable strata."},
# ---------- SOIL ----------
{"id":"soil_01","topic":"soil","climate_zone":"tropical","source":"ICAR / FAO soil health",
 "text":"Gangetic alluvial soils (Kolkata region): typically pH 6.5–7.8, low organic carbon (0.3–0.5%, target >0.75%), N deficient, P/K moderate. "
 "Improvement: green manure (Sesbania/dhaincha 45-day cycle adds 60–80 kg N/ha), compost 5–10 t/ha/yr, vermicompost for urban beds, "
 "biochar 2–5 t/ha raises CEC and water holding 15–25%, mulching cuts surface evaporation 30–50%. "
 "Urban planting pits: 1×1×1 m, refill 60% excavated soil + 30% compost + 10% sand; mycorrhizal inoculation lifts sapling survival 15–20%. "
 "Salinity (coastal/Sundarbans fringe): choose salt-tolerant Casuarina, coconut, Pongamia; gypsum amendment for sodic patches."},
{"id":"soil_02","topic":"soil","climate_zone":"global","source":"FAO Voluntary Guidelines for Soil Management",
 "text":"Soil bioengineering for slopes and bunds: Vetiver hedgerows at 1 m vertical interval; coir geotextiles with native grass seeding; "
 "avoid bare-soil monsoon exposure — cover crops always. Compacted urban soils: vertical mulching/air-spading around existing trees, "
 "structural soils (CU-Soil) under pavements give roots 20%+ void space. Phytoremediation of contaminated plots: "
 "Vetiver and Brassica juncea for heavy metals, Ricinus communis for cadmium — do not plant food species on suspect soils."},
# ---------- CARBON ----------
{"id":"carb_01","topic":"carbon","climate_zone":"global","source":"IPCC AR6 WGIII Ch.7 / i-Tree",
 "text":"Carbon sequestration (IPCC Tier-1 style): mature tropical broadleaf 20–30 kg CO2/tree/yr; fast growers (Bamboo clumps, Kadamba) 25–40; "
 "temperate broadleaf 20–30 at maturity; shrubs 1–3 kg/plant/yr. Saplings sequester ~10% of mature rate in years 1–3, 50% by year 5, "
 "full rate by years 8–12 (logistic growth curve). Apply 15% cumulative mortality in first 3 years (use 85% survival factor). "
 "Per-hectare benchmarks: tropical mixed plantation 6–12 t CO2/ha/yr at maturity; Miyawaki dense plots higher per-area in early decades. "
 "Equivalences: 1 passenger car ≈ 4.6 t CO2/yr; 1 Indian household electricity ≈ 1.5 t CO2/yr; 1 t CO2 ≈ 45 mature-tree-years."},
{"id":"carb_02","topic":"carbon","climate_zone":"global","source":"i-Tree species database",
 "text":"Species-specific annual CO2 uptake (mature, kg/tree/yr, urban open-grown): Ficus benghalensis 28; Azadirachta indica 22; Terminalia arjuna 25; "
 "Neolamarckia cadamba 24; Shorea robusta 26; Delonix regia 18; Cassia fistula 15; Lagerstroemia speciosa 14; Bambusa balcooa clump 35; "
 "Bombax ceiba 24; Artocarpus heterophyllus 21; Syzygium cumini 23; Pongamia pinnata 20; Quercus robur 30; Tilia cordata 26; "
 "Platanus × acerifolia 29; Betula pendula 18; Croton megalocarpus 25. Confidence: High when species-level value used, "
 "Medium genus-level, Low climate-zone average."},
# ---------- URBAN PLANNING ----------
{"id":"urb_01","topic":"urban","climate_zone":"global","source":"WHO Urban Green Space guidance / UN-Habitat",
 "text":"Green space norms: WHO minimum 9 m²/capita green space, ideal 50 m²; access standard — public green ≥0.5 ha within 300 m of every home (3-30-300 rule: "
 "see 3 trees from home, 30% canopy in neighbourhood, 300 m to park). Density classes: Rural <150/km², Peri-urban 150–1000, Urban 1000–5000, Dense urban >5000. "
 "Kolkata KMC density ~24,000/km², green cover ~7% (target 15%+); green space ~2 m²/capita vs WHO 9 — deficit ≈ 7 m²/person. "
 "Land that can green without displacement: road medians/verges (avenue planting), institutional campuses (schools, hospitals — typically 15–30% of urban land), "
 "industrial buffers, canal/river banks, rooftops (10–20% of plan area feasible), parking lots (40% shade mandate), cemetery/temple lands."},
{"id":"urb_02","topic":"urban","climate_zone":"global","source":"Miyawaki method / blue-green infrastructure literature",
 "text":"Dense-urban interventions: Miyawaki micro-forest — 30+ native species, 3 saplings/m², plots from 30 m² (≈90 trees); survival >90%, "
 "10x faster canopy than conventional. Green corridors along BRT/metro alignments connect fragmented habitat — target 20 m wide strips. "
 "Permeable pavement in high-footfall markets cuts runoff 70–90%. Bio-retention cells (rain gardens) every 300 m of stormwater drain: "
 "size 5–10% of contributing catchment. Trees per capita to offset residential CO2: Indian urban per-capita footprint ~2 t/yr → "
 "~90 mature trees per 1000 residents offset ~1% — so framing must be honest: urban forests are for air quality, heat and habitat first; "
 "deep decarbonisation needs energy transition. Vertical gardens on flyover pillars: 1 m² green wall ≈ 2.3 kg CO2/yr + PM capture."},
# ---------- BIODIVERSITY ----------
{"id":"bio_01","topic":"biodiversity","climate_zone":"global","source":"IUCN / India Biodiversity Portal",
 "text":"Urban biodiversity design: canopy layering — emergent (Bombax, Shorea), canopy (Ficus, Mangifera), sub-canopy (Cassia, Lagerstroemia), "
 "shrub (Hibiscus, Murraya), ground (native grasses, Curcuma). Pollinator support: continuous flowering calendar — Palash (Feb–Mar), "
 "Krishnachura (Apr–Jun), Kadamba (Jun–Aug), Cassia (Apr–Jul), Shiuli/Nyctanthes (Sep–Nov). Keystone figs (F. benghalensis, F. religiosa) "
 "support 100+ vertebrate species. Wetland birds need shallow-margin zones (<30 cm) and emergent reeds. Dead wood retention and "
 "no-mow meadow patches raise urban invertebrate abundance 3–5x. Connectivity: stepping-stone pocket parks every 500 m enable bird/butterfly movement."},
# ---------- RENEWABLE ENERGY ----------
{"id":"energy_01","topic":"energy","climate_zone":"global","source":"IRENA / MNRE siting guidelines",
 "text":"Renewable siting feasibility rules: ground-mounted solar needs slope <5° ideal, <10° maximum — on steep or "
 "mountainous terrain grading causes erosion and landslide risk, so use rooftop and canopy solar only. NEVER clear forest "
 "or green cover for solar farms (carbon payback becomes negative for decades); prefer degraded/barren land, brownfields, "
 "capped landfills, parking canopies and canal-top arrays (canal-top ≈1 MWp per km of 10 m canal, plus evaporation savings). "
 "Agrivoltaics keeps farmland productive while panels reduce crop heat stress 1-3°C. Typical yields: 1 kWp ≈ insolation × 0.75 "
 "performance ratio × 365 kWh/yr; 1 ha ≈ 0.8 MWp. Small wind viable at mean speeds ≥4 m/s, good ≥5.5 m/s; urban turbulence "
 "favours rooftop-edge or open-field siting. Micro-hydro needs ≥60 m relief per km and ≥1000 mm rainfall with perennial flow. "
 "Community biogas: 0.3-0.5 kg organic waste/person/day, 40-60 m³ biogas per tonne wet waste, digestate is fertiliser. "
 "Solar irrigation pumps replace diesel: ~2-3 t CO2/yr per 5 HP pump. Grid emission factors (kg CO2/kWh): India 0.71, "
 "UK 0.21, Germany 0.38, US 0.37, Kenya 0.10, Brazil 0.10, France 0.06, UAE 0.49."},
# ---------- REGULATORY ----------
{"id":"reg_01","topic":"regulation","climate_zone":"global","source":"India environmental framework",
 "text":"India regulatory context: EIA Notification 2006 (construction >20,000 m² needs clearance); Wetlands Rules 2017 (no encroachment/solid waste in notified wetlands — "
 "East Kolkata Wetlands are Ramsar-protected); CRZ rules for coastal belts; Tree Acts require permission + compensatory planting (typ. 1:5) for felling; "
 "NCAP targets 40% PM reduction by 2026 in non-attainment cities (Kolkata included); Jal Shakti Abhiyan promotes RWH; "
 "Smart Cities/AMRUT fund green-blue infrastructure; municipal green budget benchmark 2–5% of capex. NGT precedents protect urban water bodies from landfill."},
]
print(f"Knowledge base: {len(KNOWLEDGE_DOCS)} curated documents, "
      f"topics: {sorted(set(d['topic'] for d in KNOWLEDGE_DOCS))}")


In [ ]:
# ============================================================
# Cell 5: RAG Pipeline — hybrid retrieval with graceful fallback
#   dense (ChromaDB + bge-small) ∪ sparse (BM25) → fused rerank
#   Fallback chain: Chroma+ST → BM25 only → pure TF-IDF (stdlib)
# ============================================================
import collections

def _chunk(text, size=512, overlap=50):
    words = text.split()
    step = max(size - overlap, 1)
    return [" ".join(words[i:i+size]) for i in range(0, len(words), step)] or [text]

CHUNKS = []
for d in KNOWLEDGE_DOCS:
    for j, ch in enumerate(_chunk(d["text"])):
        CHUNKS.append({"id": f"{d['id']}_{j}", "text": ch,
                       "metadata": {"source": d["source"], "topic": d["topic"],
                                    "climate_zone": d["climate_zone"]}})

_tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())

class HybridRetriever:
    """Dense + sparse hybrid with reciprocal-rank fusion."""
    def __init__(self, chunks):
        self.chunks = chunks
        self.mode = []
        # --- dense: ChromaDB + sentence-transformers ---
        self.collection = None
        if CAPS["chromadb"] and CAPS["sentence_transformers"]:
            try:
                import chromadb
                from sentence_transformers import SentenceTransformer
                self.embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
                client = chromadb.PersistentClient(path=VECTORDB_PATH)
                try: client.delete_collection("ecogpt_knowledge")
                except Exception: pass
                self.collection = client.get_or_create_collection("ecogpt_knowledge")
                self.collection.add(
                    ids=[c["id"] for c in chunks],
                    embeddings=self.embedder.encode([c["text"] for c in chunks],
                                                    show_progress_bar=False).tolist(),
                    documents=[c["text"] for c in chunks],
                    metadatas=[c["metadata"] for c in chunks])
                self.mode.append("dense:chromadb+bge-small")
            except Exception as e:
                print(f"  dense retrieval unavailable ({e}); continuing with sparse")
                self.collection = None
        # --- sparse: BM25 or TF-IDF ---
        self.bm25 = None
        if CAPS["rank_bm25"]:
            self.bm25 = BM25Okapi([_tok(c["text"]) for c in chunks])
            self.mode.append("sparse:bm25")
        else:  # stdlib TF-IDF
            self.df = collections.Counter()
            self.doc_tokens = [collections.Counter(_tok(c["text"])) for c in chunks]
            for tc in self.doc_tokens: self.df.update(set(tc))
            self.N = len(chunks)
            self.mode.append("sparse:tfidf-fallback")

    def _sparse_scores(self, query):
        q = _tok(query)
        if self.bm25 is not None:
            return list(self.bm25.get_scores(q))
        scores = []
        for tc in self.doc_tokens:
            L = sum(tc.values()) or 1
            s = sum((tc[t]/L) * math.log(1 + self.N/(1 + self.df.get(t, 0))) for t in q)
            scores.append(s)
        return scores

    def retrieve(self, query, n_results=5, topic=None):
        ranks = []
        if self.collection is not None:
            res = self.collection.query(
                query_embeddings=self.embedder.encode([query]).tolist(),
                n_results=min(n_results*3, len(self.chunks)))
            id2idx = {c["id"]: i for i, c in enumerate(self.chunks)}
            ranks.append([id2idx[i] for i in res["ids"][0]])
        sp = self._sparse_scores(query)
        ranks.append(sorted(range(len(sp)), key=lambda i: -sp[i])[:n_results*3])
        # reciprocal-rank fusion
        fused = collections.defaultdict(float)
        for rank_list in ranks:
            for r, idx in enumerate(rank_list):
                fused[idx] += 1.0 / (60 + r)
        order = sorted(fused, key=lambda i: -fused[i])
        out = []
        for idx in order:
            c = self.chunks[idx]
            if topic and c["metadata"]["topic"] != topic and fused[idx] < 0.02:
                continue
            out.append({"id": c["id"], "text": c["text"], "source": c["metadata"]["source"],
                        "topic": c["metadata"]["topic"], "relevance": round(fused[idx], 4),
                        "query": query})
            if len(out) >= n_results: break
        if not out:
            return [{"id": None, "text": "No relevant knowledge found for this query.",
                     "source": "none", "topic": topic, "relevance": 0.0, "query": query}]
        return out

retriever = HybridRetriever(CHUNKS)
print(f"Retriever ready | mode: {' + '.join(retriever.mode)} | {len(CHUNKS)} chunks")

def retrieve(query: str, n_results: int = 5, topic: str = None) -> list:
    """RAG retrieval tool — returns chunks with source, relevance and query."""
    return retriever.retrieve(query, n_results, topic)

def build_retrieval_query(agent_type: str, context: dict) -> str:
    loc, aqi = context.get("location", {}), context.get("aqi", {})
    queries = {
        "pollution": f"pollution mitigation strategies AQI {aqi.get('category','')} {loc.get('climate_zone','')} heat island actions",
        "plantation": f"native tree species {loc.get('city','')} {loc.get('climate_zone','')} planting guide carbon biofilter",
        "water": f"water conservation rainwater harvesting restoration {loc.get('climate_zone','')} humidity {context.get('humidity_avg','')}",
        "soil": "soil improvement organic matter compost planting pit tropical urban",
        "carbon": "carbon sequestration trees CO2 uptake rates mortality growth curve equivalences",
        "urban": f"urban green space planning population density {context.get('population_density','')} per km2 miyawaki corridors",
        "biodiversity": f"biodiversity canopy layers pollinators {loc.get('climate_zone','')}",
        "regulation": f"environmental regulation {loc.get('country','India')} wetlands tree act clean air",
    }
    return queries.get(agent_type, f"environmental sustainability {loc.get('city','')}")

# smoke test
for r in retrieve("native trees for tropical monsoon Kolkata", 2):
    print(f"  [{r['relevance']}] {r['id']} ({r['source']})")


In [ ]:
# ============================================================
# Cell 6: Tool Definitions
#   AQI calculator (EPA breakpoints) | reverse geocode (online +
#   offline gazetteer) | Köppen climate zone | population density |
#   species DB + filter | sensor loader | what-if simulation
# ============================================================

# ---------- EPA AQI ----------
AQI_BREAKPOINTS = {
    "PM2.5": [(0.0,12.0,0,50),(12.1,35.4,51,100),(35.5,55.4,101,150),
              (55.5,150.4,151,200),(150.5,250.4,201,300),(250.5,500.4,301,500)],
    "CO":    [(0.0,4.4,0,50),(4.5,9.4,51,100),(9.5,12.4,101,150),
              (12.5,15.4,151,200),(15.5,30.4,201,300),(30.5,50.4,301,500)],
    "NO2":   [(0,53,0,50),(54,100,51,100),(101,360,101,150),
              (361,649,151,200),(650,1249,201,300),(1250,2049,301,500)],
}
AQI_CATEGORIES = [(50,"Good"),(100,"Moderate"),(150,"Unhealthy for Sensitive Groups"),
                  (200,"Unhealthy"),(300,"Very Unhealthy"),(10**9,"Hazardous")]

def aqi_subindex(pollutant: str, concentration: float) -> float:
    """EPA linear interpolation. CO in ppm, NO2 in ppb, PM2.5 in µg/m³."""
    if concentration is None or (isinstance(concentration, float) and math.isnan(concentration)):
        return float("nan")
    bps = AQI_BREAKPOINTS[pollutant]
    c = min(max(concentration, 0), bps[-1][1])
    for lo, hi, ilo, ihi in bps:
        if c <= hi:  # clamp into segment — EPA tables have gaps (e.g. CO 4.4→4.5)
            return (ihi-ilo)/(hi-lo)*(max(c, lo)-lo)+ilo
    return 500.0

def aqi_category(score: float) -> str:
    for ceiling, name in AQI_CATEGORIES:
        if score <= ceiling: return name
    return "Hazardous"

def compute_aqi(no2_ppm=None, co_ppm=None, pm25=None, mq135=None, dd=None) -> dict:
    """Composite AQI = max of sub-indices (EPA convention).
    NO2 sensor reads ppm → ppb. If RAWPM missing, fall back to dust density
    (DD) scaled to a PM2.5 proxy (~0.4 fine fraction), flagged as estimated."""
    subs, est = {}, []
    if pm25 is not None and not math.isnan(pm25 if pm25 is not None else float("nan")):
        subs["PM2.5"] = aqi_subindex("PM2.5", pm25)
    elif dd is not None and not math.isnan(dd):
        subs["PM2.5"] = aqi_subindex("PM2.5", dd*0.4); est.append("PM2.5 from dust density (estimated)")
    if co_ppm is not None and not math.isnan(co_ppm):
        # MQ-7 is a ratio-type sensor: sustained readings >30 ppm are not credible
        # calibrated ambient CO (30+ ppm ≈ occupational limit). Treat as uncalibrated
        # ratio and rescale ÷10, flagged as an assumption.
        if co_ppm > 30:
            co_ppm = co_ppm / 10.0
            est.append("MQ7 CO channel treated as uncalibrated ratio, rescaled ÷10 (assumed)")
        subs["CO"] = aqi_subindex("CO", co_ppm)
    if no2_ppm is not None and not math.isnan(no2_ppm):
        # Ambient NO2 rarely exceeds 0.2 ppm; electrochemical channel readings >1 ppm
        # indicate raw/uncalibrated output → rescale ÷100, flagged.
        if no2_ppm > 1:
            no2_ppm = no2_ppm / 100.0
            est.append("NO2 channel rescaled ÷100 to plausible ambient range (assumed)")
        subs["NO2"] = aqi_subindex("NO2", no2_ppm*1000)  # ppm → ppb
    if mq135 is not None and not math.isnan(mq135) and mq135 > 9:
        est.append("MQ135 elevated — broad-spectrum gas load corroborates pollution")
    if not subs:
        return {"score": float("nan"), "category": "Unknown", "dominant": None, "notes": ["no usable pollutant data"]}
    dom = max(subs, key=subs.get)
    score = round(subs[dom], 1)
    return {"score": score, "category": aqi_category(score), "dominant": dom,
            "sub_indices": {k: round(v,1) for k,v in subs.items()}, "notes": est}

# ---------- Offline gazetteer (works with zero network) ----------
GAZETTEER = {  # name: (lat, lon, country, köppen, pop_density/km², annual_rain_mm)
 "kolkata": (22.5726, 88.3639, "India", "Aw", 24000, 1800),
 "delhi": (28.6139, 77.2090, "India", "BSh", 11300, 800),
 "mumbai": (19.0760, 72.8777, "India", "Aw", 21000, 2200),
 "chennai": (13.0827, 80.2707, "India", "Aw", 26900, 1400),
 "bengaluru": (12.9716, 77.5946, "India", "Aw", 11900, 980),
 "hyderabad": (17.3850, 78.4867, "India", "BSh", 18500, 800),
 "sundarbans": (21.9497, 88.9468, "India", "Aw", 120, 1900),
 "dhaka": (23.8103, 90.4125, "Bangladesh", "Aw", 23000, 2000),
 "london": (51.5074, -0.1278, "United Kingdom", "Cfb", 5700, 600),
 "new york": (40.7128, -74.0060, "United States", "Cfa", 11000, 1200),
 "nairobi": (-1.2921, 36.8219, "Kenya", "Aw", 6200, 870),
 "lagos": (6.5244, 3.3792, "Nigeria", "Aw", 13100, 1700),
 "singapore": (1.3521, 103.8198, "Singapore", "Af", 8000, 2340),
 "dubai": (25.2048, 55.2708, "UAE", "BWh", 1100, 100),
 "tokyo": (35.6762, 139.6503, "Japan", "Cfa", 6400, 1530),
 "sao paulo": (-23.5505, -46.6333, "Brazil", "Cfa", 7900, 1450),
 "berlin": (52.5200, 13.4050, "Germany", "Dfb", 4100, 570),
 "sydney": (-33.8688, 151.2093, "Australia", "Cfa", 430, 1210),
 "cairo": (30.0444, 31.2357, "Egypt", "BWh", 19000, 25),
 "moscow": (55.7558, 37.6173, "Russia", "Dfb", 5000, 700),
}

def get_climate_zone(lat: float, lon: float) -> str:
    """Köppen approximation: nearest gazetteer city <500 km, else latitude bands."""
    best, bd = None, 1e9
    for name,(la,lo,co,kz,pd_,rain) in GAZETTEER.items():
        d = ((lat-la)**2 + (lon-lo)**2)**0.5
        if d < bd: best, bd = kz, d
    if bd < 4.5: return best
    a = abs(lat)
    if a < 10: return "Af"
    if a < 23.5: return "Aw"
    if a < 35: return "Cfa"
    if a < 55: return "Cfb"
    return "Dfb"

CLIMATE_ZONE_NAMES = {"Af":"Tropical rainforest","Am":"Tropical monsoon","Aw":"Tropical savanna/wet-dry",
 "BWh":"Hot desert","BSh":"Hot semi-arid","Cfa":"Humid subtropical","Cfb":"Temperate oceanic",
 "Dfb":"Warm-summer continental"}
ZONE_TO_SPECIES_KEY = {"Af":"tropical","Am":"tropical","Aw":"tropical","BWh":"arid","BSh":"arid",
 "Cfa":"temperate","Cfb":"temperate","Dfb":"temperate"}

def reverse_geocode(lat: float, lon: float) -> dict:
    """Online Nominatim when available; offline gazetteer fallback otherwise."""
    if CAPS["geopy"]:
        try:
            geo = Nominatim(user_agent="ecogpt_hackathon", timeout=5)
            loc = geo.reverse((lat, lon), language="en", zoom=10)
            if loc:
                a = loc.raw.get("address", {})
                city = a.get("city") or a.get("town") or a.get("village") or a.get("county") or "Unknown"
                return {"city": city, "district": a.get("state_district", ""),
                        "country": a.get("country", ""), "lat": lat, "lon": lon,
                        "climate_zone": get_climate_zone(lat, lon), "geocoder": "nominatim"}
        except Exception:
            pass
    best, bd = "Unknown", 1e9
    for name,(la,lo,co,kz,pd_,rain) in GAZETTEER.items():
        d = ((lat-la)**2+(lon-lo)**2)**0.5
        if d < bd: best, bd, country = name.title(), d, co
    return {"city": best if bd < 2 else f"({lat:.3f}, {lon:.3f})",
            "district": "", "country": country if bd < 2 else "Unknown",
            "lat": lat, "lon": lon, "climate_zone": get_climate_zone(lat, lon),
            "geocoder": "offline-gazetteer"}

def geocode_city(name: str):
    """City name → (lat, lon). Gazetteer first, then Nominatim."""
    key = name.strip().lower()
    for g, vals in GAZETTEER.items():
        if g in key or key in g: return vals[0], vals[1]
    if CAPS["geopy"]:
        try:
            loc = Nominatim(user_agent="ecogpt_hackathon", timeout=5).geocode(name)
            if loc: return loc.latitude, loc.longitude
        except Exception: pass
    return None

def get_open_spaces(lat: float, lon: float, radius_m: int = 3000) -> list:
    """REAL plantable-land lookup via OpenStreetMap Overpass API:
    parks, meadows, brownfields, vacant/green land + water bodies near the point.
    Returns [{name, kind, coords[(lat,lon)…], area_m2, tree_capacity}]. [] offline."""
    q = f"""[out:json][timeout:25];(
      way["leisure"~"park|recreation_ground|garden|pitch"](around:{radius_m},{lat},{lon});
      way["landuse"~"grass|meadow|brownfield|greenfield|village_green|recreation_ground|cemetery|allotments"](around:{radius_m},{lat},{lon});
      way["natural"~"^water$|wetland|scrub|grassland"](around:{radius_m},{lat},{lon});
    );out geom 120;"""
    try:
        r = _rq_get_post("https://overpass-api.de/api/interpreter", q)
        elements = r.json().get("elements", [])
    except Exception:
        return []
    out = []
    for el in elements:
        geom = el.get("geometry") or []
        if len(geom) < 4: continue
        coords = [(p["lat"], p["lon"]) for p in geom]
        tags = el.get("tags", {})
        kind = ("water" if tags.get("natural") in ("water", "wetland") else "open")
        area = _poly_area_m2(coords)
        if area < 400: continue            # skip tiny slivers
        out.append({"name": tags.get("name", tags.get("landuse") or tags.get("leisure")
                    or tags.get("natural") or "unnamed plot"),
                    "kind": kind, "coords": coords, "area_m2": int(area),
                    "tree_capacity": 0 if kind == "water" else int(area/25)})  # 1 tree/25 m² urban spacing
    out.sort(key=lambda s: -s["area_m2"])
    return out[:60]

def _rq_get_post(url, data):
    import requests as _r
    return _r.post(url, data={"data": data}, timeout=30,
                   headers={"User-Agent": "ecogpt_hackathon"})

def _poly_area_m2(coords):
    """Shoelace area for small lat/lon polygons."""
    if len(coords) < 3: return 0.0
    R, lat0 = 6371000.0, math.radians(coords[0][0])
    pts = [(math.radians(lo)*R*math.cos(lat0), math.radians(la)*R) for la, lo in coords]
    s = sum(x1*y2 - x2*y1 for (x1, y1), (x2, y2) in zip(pts, pts[1:] + pts[:1]))
    return abs(s)/2

# ---------- LIVE per-location data (Open-Meteo: keyless, global, free) ----------
_LIVE_CACHE, _TERRAIN_CACHE, _POP_CACHE = {}, {}, {}

def fetch_live_environment(lat: float, lon: float):
    """REAL location-specific data for ANY point on Earth:
    CAMS air quality (PM2.5/PM10/CO/NO2) + ERA5 weather + solar radiation + wind.
    Returns dict or None when offline."""
    key = (round(lat, 2), round(lon, 2))
    if key in _LIVE_CACHE: return _LIVE_CACHE[key]
    import requests as _r
    try:
        aq = _r.get("https://air-quality-api.open-meteo.com/v1/air-quality",
            params={"latitude": lat, "longitude": lon, "past_days": 2,
                    "hourly": "pm2_5,pm10,carbon_monoxide,nitrogen_dioxide"},
            timeout=12).json()["hourly"]
        wx = _r.get("https://api.open-meteo.com/v1/forecast",
            params={"latitude": lat, "longitude": lon, "past_days": 7,
                    "hourly": "temperature_2m,relative_humidity_2m,apparent_temperature,"
                              "shortwave_radiation,wind_speed_10m"},
            timeout=12).json()["hourly"]
        m = lambda v: float(np.nanmean([x for x in v if x is not None])) if any(
            x is not None for x in v) else float("nan")
        out = {"pm25": m(aq["pm2_5"]), "pm10": m(aq["pm10"]),
               "co_ppm": m(aq["carbon_monoxide"]) / 1145.0,    # µg/m³ → ppm (25°C)
               "no2_ppb": m(aq["nitrogen_dioxide"]) / 1.88,    # µg/m³ → ppb
               "tmp": m(wx["temperature_2m"]), "hmd": m(wx["relative_humidity_2m"]),
               "hi": m(wx["apparent_temperature"]),
               "solar_kwh_m2_day": m(wx["shortwave_radiation"]) * 24 / 1000.0,
               "wind_ms": m(wx["wind_speed_10m"]) / 3.6}       # km/h → m/s
        _LIVE_CACHE[key] = out if not math.isnan(out["tmp"]) else None
    except Exception:
        _LIVE_CACHE[key] = None
    return _LIVE_CACHE[key]

def get_terrain(lat: float, lon: float):
    """Slope/relief from Open-Meteo elevation API (9-point ~1 km grid).
    Drives feasibility: no ground-mounted solar / heavy works on steep terrain."""
    key = (round(lat, 2), round(lon, 2))
    if key in _TERRAIN_CACHE: return _TERRAIN_CACHE[key]
    import requests as _r
    try:
        d = 0.009
        pts = [(lat,lon),(lat+d,lon),(lat-d,lon),(lat,lon+d),(lat,lon-d),
               (lat+d,lon+d),(lat-d,lon-d),(lat+d,lon-d),(lat-d,lon+d)]
        r = _r.get("https://api.open-meteo.com/v1/elevation",
                   params={"latitude": ",".join(str(p[0]) for p in pts),
                           "longitude": ",".join(str(p[1]) for p in pts)}, timeout=10).json()
        el = r["elevation"]; relief = max(el) - min(el)
        slope = ("flat" if relief < 15 else "rolling" if relief < 60
                 else "steep" if relief < 150 else "mountainous")
        _TERRAIN_CACHE[key] = {"elevation_m": round(el[0]), "relief_m_per_km": round(relief),
                               "slope_class": slope}
    except Exception:
        _TERRAIN_CACHE[key] = None
    return _TERRAIN_CACHE[key]

ZONE_SOLAR = {"Af":4.5,"Am":4.8,"Aw":5.2,"BWh":6.3,"BSh":5.8,"Cfa":4.3,"Cfb":2.9,"Dfb":3.2}

def get_population_density(lat: float, lon: float) -> float:
    """persons/km² — REAL data first: nearest OSM place node with a population
    tag (Overpass) → place-type typical density → gazetteer anchor → rural 300."""
    key = (round(lat, 2), round(lon, 2))
    if key in _POP_CACHE: return _POP_CACHE[key]
    dens = None
    try:
        q = (f'[out:json][timeout:15];node["place"~"city|town|suburb|village"]'
             f'(around:12000,{lat},{lon});out 30;')
        els = _rq_get_post("https://overpass-api.de/api/interpreter", q).json()["elements"]
        best = None
        for e in els:
            t = e.get("tags", {})
            score = (0 if t.get("population") else 1,
                     {"city":0,"suburb":1,"town":2,"village":3}.get(t.get("place"), 4))
            if best is None or score < best[0]: best = (score, t)
        if best:
            t = best[1]; place = t.get("place", "town")
            if t.get("population"):
                pop = float(re.sub(r"[^\d.]", "", t["population"]) or 0)
                typ_area = {"city":120,"suburb":10,"town":25,"village":6}.get(place, 25)
                dens = max(pop/typ_area, 50)
            else:
                dens = {"city":8000,"suburb":9000,"town":2500,"village":400}.get(place, 1000)
    except Exception:
        pass
    if dens is None:  # offline → gazetteer anchor with decay
        best, bd = 300.0, 1e9
        for name,(la,lo,co,kz,d_,rain) in GAZETTEER.items():
            dd_ = ((lat-la)**2+(lon-lo)**2)**0.5
            if dd_ < bd: best, bd = d_, dd_
        dens = best if bd < 0.15 else best*max(0.15, 1-bd/1.5) if bd < 1.5 else 300.0
    _POP_CACHE[key] = float(dens)
    return float(dens)

def get_annual_rainfall(lat: float, lon: float) -> float:
    best, bd = 1000.0, 1e9
    for name,(la,lo,co,kz,dens,rain) in GAZETTEER.items():
        d = ((lat-la)**2+(lon-lo)**2)**0.5
        if d < bd: best, bd = rain, d
    return best

def classify_density(d: float) -> str:
    return ("Rural" if d < 150 else "Peri-urban" if d < 1000 else
            "Urban" if d < 5000 else "Dense Urban")

# ---------- Species database ----------
SPECIES_DB = [ # name, common, zone, role, co2 kg/yr, water, native_to, season, growth, layer
 ("Ficus benghalensis","Banyan","tropical","Keystone shade, carbon sink, bird habitat",28,"medium","India","Jun–Jul","slow-massive","canopy"),
 ("Azadirachta indica","Neem","tropical","Air purifier (NO2/SO2), medicinal, drought-hardy",22,"low","India","Jun–Aug","medium","canopy"),
 ("Terminalia arjuna","Arjun","tropical","Riparian stabiliser, carbon sink",25,"high","India","Jun–Jul","medium","canopy"),
 ("Neolamarckia cadamba","Kadamba","tropical","Fast carbon, pollinator support",24,"medium","India","Jun–Jul","fast","canopy"),
 ("Shorea robusta","Sal","tropical","Forest restoration dominant",26,"medium","India","Jun–Jul","slow","emergent"),
 ("Madhuca longifolia","Mahua","tropical","Livelihood, pollinator, dry-tolerant",21,"low","India","Jun–Jul","slow","canopy"),
 ("Butea monosperma","Palash","tropical","Pollinator (Feb–Mar bloom), N-fixing",17,"low","India","Jul–Aug","slow","sub-canopy"),
 ("Delonix regia","Krishnachura/Gulmohar","tropical","Avenue flowering, heat tolerant",18,"low","naturalized","Jun–Jul","fast","sub-canopy"),
 ("Bombax ceiba","Shimul/Silk cotton","tropical","Emergent layer, bird nesting",24,"medium","India","Jun–Jul","fast","emergent"),
 ("Artocarpus heterophyllus","Jackfruit","tropical","Food security + shade",21,"medium","India","Jun–Aug","medium","canopy"),
 ("Syzygium cumini","Jamun","tropical","Riparian, edible, dense shade",23,"medium","India","Jun–Jul","medium","canopy"),
 ("Bambusa balcooa","Bamboo (clumping)","tropical","Fastest carbon, soil binder, NOT for monoculture",35,"medium","India","Jun–Jul","very fast","sub-canopy"),
 ("Cassia fistula","Amaltas","tropical","Compact avenue tree for dense urban",15,"low","India","Jun–Jul","medium","sub-canopy"),
 ("Lagerstroemia speciosa","Pride of India/Jarul",ZONE_TO_SPECIES_KEY["Aw"],"Compact flowering, waterlogging-tolerant",14,"medium","India","Jun–Jul","medium","sub-canopy"),
 ("Pongamia pinnata","Karanja","tropical","N-fixing, biodiesel, coastal-tolerant",20,"low","India","Jun–Aug","medium","canopy"),
 ("Chrysopogon zizanioides","Vetiver grass","tropical","Bund/slope erosion control, phytoremediation",2,"low","India","Jun–Sep","fast","ground"),
 ("Nelumbo nucifera","Lotus","tropical","Aquatic bioremediation, biodiversity",1,"aquatic","India","Mar–Jun","fast","aquatic"),
 ("Typha angustifolia","Cattail/Hogla","tropical","Reed-bed water polishing",2,"aquatic","India","Jun–Sep","fast","aquatic"),
 ("Prosopis cineraria","Khejri","arid","Desert keystone, N-fixing fodder",15,"very low","India","Jul–Aug","slow","canopy"),
 ("Moringa oleifera","Drumstick","arid","Fast nutrition tree, drought-hardy",12,"very low","India","Jun–Aug","very fast","sub-canopy"),
 ("Ziziphus mauritiana","Ber/Indian jujube","arid","Fruit, hardy, pollinator",13,"very low","India","Jul–Aug","medium","sub-canopy"),
 ("Vachellia nilotica","Babul","arid","N-fixing, gum, fodder",16,"very low","India","Jul–Aug","medium","canopy"),
 ("Quercus robur","English oak","temperate","Keystone biodiversity (2300+ spp.), carbon",30,"medium","Europe","Nov–Mar","slow","canopy"),
 ("Tilia cordata","Small-leaved lime","temperate","Pollinator keystone, avenue",26,"medium","Europe","Nov–Mar","medium","canopy"),
 ("Platanus x acerifolia","London plane","temperate","Pollution-tolerant avenue standard",29,"medium","naturalized","Nov–Mar","fast","canopy"),
 ("Betula pendula","Silver birch","temperate","PM capture (leaf hairs), pioneer",18,"medium","Europe","Nov–Mar","fast","sub-canopy"),
 ("Sorbus aucuparia","Rowan","temperate","Bird forage, compact",12,"medium","Europe","Nov–Mar","medium","sub-canopy"),
 ("Croton megalocarpus","Croton","tropical","Highland shade + carbon (East Africa)",25,"medium","East Africa","Mar–May","fast","canopy"),
 ("Markhamia lutea","Nile tulip","tropical","Highland avenue, timber (East Africa)",18,"low","East Africa","Mar–May","fast","sub-canopy"),
 ("Chlorophytum comosum","Spider plant","global","Indoor/vertical-garden VOC removal",1,"low","Africa","any","fast","ground"),
 ("Spathiphyllum wallisii","Peace lily","global","Indoor VOC/formaldehyde removal",1,"medium","Americas","any","medium","ground"),
 ("Hedera helix","English ivy","temperate","NO2-absorbing green screens",2,"low","Europe","Sep–Nov","fast","ground"),
]
INVASIVE_BLACKLIST = ["Prosopis juliflora","Lantana camara","Eichhornia crassipes",
                      "Parthenium hysterophorus","Leucaena leucocephala","Acacia mearnsii"]

COUNTRY_TO_REGION = {"india":"India","bangladesh":"India","pakistan":"India","nepal":"India",
 "sri lanka":"India","kenya":"East Africa","ethiopia":"East Africa","tanzania":"East Africa",
 "united kingdom":"Europe","germany":"Europe","france":"Europe","russia":"Europe"}

def filter_species_by_climate(climate_zone: str, density_class: str, n: int = 12,
                              objective: str = "general", country: str = "") -> list:
    """Return suitable species records for zone + density, invasive-checked,
    preferring species native to the detected region."""
    key = ZONE_TO_SPECIES_KEY.get(climate_zone, "tropical")
    rows = [s for s in SPECIES_DB if s[2] in (key, "global")]
    region = COUNTRY_TO_REGION.get((country or "").strip().lower(), None)
    def native_penalty(s):
        if region is None or s[6] in ("naturalized", "global"): return 0
        return 0 if s[6] == region else 1
    if density_class == "Dense Urban":
        pref = {"sub-canopy","ground","aquatic"}
        rows.sort(key=lambda s: (native_penalty(s), s[9] not in pref, -s[4]))
    elif density_class == "Rural":
        rows.sort(key=lambda s: (native_penalty(s), s[9] not in {"emergent","canopy"}, -s[4]))
    else:
        rows.sort(key=lambda s: (native_penalty(s), -s[4]))
    if objective == "water":
        rows = [s for s in rows if s[9] in {"aquatic","ground"} or s[5] in {"high","aquatic"}] + rows
    rows = [s for s in rows if s[0] not in INVASIVE_BLACKLIST]
    seen, out = set(), []
    for s in rows:
        if s[0] in seen: continue
        seen.add(s[0]); out.append(s)
        if len(out) >= n: break
    return [dict(zip(["scientific","common","zone","role","co2_kg_yr","water",
                      "native_to","season","growth","layer"], s)) for s in out]

# ---------- Sensor loader ----------
def load_sensor_data(lat: float, lon: float, radius_deg: float = 0.05,
                     df: 'pd.DataFrame' = None) -> 'pd.DataFrame':
    df = sensor_df if df is None else df
    m = (df["LAT"].sub(lat).abs() <= radius_deg) & (df["LON"].sub(lon).abs() <= radius_deg)
    return df[m].copy()

# ---------- What-if simulation engine ----------
TOTAL_URBAN_AREA_HA = 20500  # KMC area ≈ 205 km²; overridden per location below

def simulate_intervention(baseline_aqi: float, trees_to_plant: int, area_hectares: float,
                          years: int, water_harvesting_m3: float = 0,
                          urban_area_ha: float = TOTAL_URBAN_AREA_HA) -> dict:
    """Project outcomes of a hypothetical intervention (medium confidence).
    AQI: ~0.3 points per 100 mature trees (urban canopy literature), capped at 30%.
    CO2: 22 kg/tree/yr default × growth ramp × 85% survival."""
    ramp = sum(min(0.1 + 0.1*y, 1.0) for y in range(years))   # sapling growth curve
    aqi_red = min((trees_to_plant/100)*0.3*min(years,10), baseline_aqi*0.30)
    co2_kg = trees_to_plant * 22 * ramp * 0.85
    return {"projected_aqi": round(max(0, baseline_aqi - aqi_red), 1),
            "aqi_reduction": round(aqi_red, 1),
            "co2_captured_tonnes": round(co2_kg/1000, 1),
            "cars_equivalent": int(co2_kg/1000/4.6/max(years,1)),
            "green_cover_increase_pct": round(area_hectares/urban_area_ha*100, 2),
            "water_saved_m3": int(water_harvesting_m3*years),
            "confidence": "medium"}

print("Tools ready ✓  (AQI, geocode, climate, population, species filter, loader, simulator)")
print("  e.g. compute_aqi(no2_ppm=0.04, co_ppm=8, dd=250) →", compute_aqi(no2_ppm=0.04, co_ppm=8, dd=250))


In [ ]:
# ============================================================
# Cell 7: Pluggable LLM Backend — Ollama → HuggingFace → None
#   Every agent works WITHOUT an LLM (deterministic scientific core);
#   when a backend is live the LLM polishes and calibrates narrative.
# ============================================================
import requests as _rq

OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://localhost:11434")
OLLAMA_MODELS = ["mistral:7b-instruct", "mistral", "llama3.1:8b", "llama3.1", "llama3"]
HF_TOKEN = os.environ.get("HF_TOKEN", "")
HF_MODEL = os.environ.get("HF_MODEL", "mistralai/Mistral-7B-Instruct-v0.3")

class LLMBackend:
    """Auto-detects best available backend. .generate(system, user) → str|None"""
    def __init__(self):
        self.kind, self.model = "none", None
        try:  # 1) Ollama (AMD ROCm-accelerated locally)
            tags = _rq.get(f"{OLLAMA_URL}/api/tags", timeout=2).json()
            available = [m["name"] for m in tags.get("models", [])]
            for want in OLLAMA_MODELS:
                hit = next((a for a in available if a.startswith(want)), None)
                if hit: self.kind, self.model = "ollama", hit; break
            if self.kind == "none" and available:
                self.kind, self.model = "ollama", available[0]
        except Exception:
            pass
        if self.kind == "none" and HF_TOKEN:  # 2) HuggingFace Inference API
            self.kind, self.model = "hf", HF_MODEL
        print(f"LLM backend: {self.kind}" + (f" ({self.model})" if self.model else
              "  → deterministic engine only (fully functional, narrative un-polished)"))

    def generate(self, system: str, user: str, max_tokens=900, temperature=0.4):
        try:
            if self.kind == "ollama":
                r = _rq.post(f"{OLLAMA_URL}/api/chat", json={
                    "model": self.model, "stream": False,
                    "messages": [{"role":"system","content":system},
                                 {"role":"user","content":user}],
                    "options": {"temperature": temperature, "num_predict": max_tokens}},
                    timeout=180)
                return r.json()["message"]["content"].strip()
            if self.kind == "hf":
                r = _rq.post(
                    f"https://api-inference.huggingface.co/models/{self.model}/v1/chat/completions",
                    headers={"Authorization": f"Bearer {HF_TOKEN}"},
                    json={"model": self.model, "max_tokens": max_tokens,
                          "temperature": temperature,
                          "messages": [{"role":"system","content":system},
                                       {"role":"user","content":user}]},
                    timeout=120)
                return r.json()["choices"][0]["message"]["content"].strip()
        except Exception as e:
            print(f"  (LLM call failed: {e} — using deterministic output)")
        return None

LLM = LLMBackend()


In [ ]:
# ============================================================
# Cell 8: The 8 EcoGPT Agents
#   Each agent = system prompt (for LLM/ADK use) + deterministic
#   scientific core (always runs) + RAG grounding.
# ============================================================

SYSTEM_PROMPTS = {
"data_ingestion": """You are the Data Ingestion Agent for EcoGPT. Accept a latitude/longitude or city name.
Load and filter sensor CSV data within a 0.05° radius. Compute mean/max/min/std per sensor channel.
Derive composite AQI from NO2, CO, PM (RAWPM or dust-density proxy) and MQ135 using EPA breakpoints.
Classify AQI: Good 0-50, Moderate 51-100, USG 101-150, Unhealthy 151-200, Very Unhealthy 201-300, Hazardous >300.
Flag anomalies >2σ above mean. Return structured JSON. State assumptions clearly when data is sparse.""",
"rag_retrieval": """You are the RAG Retrieval Agent for EcoGPT. Given environmental context, retrieve top-K relevant
chunks covering species, pollution mitigation, water management, soil, urban greening, carbon data, and regulation.
Always return source, relevance score, and the retrieving query. Never hallucinate; say explicitly if nothing is found.""",
"pollution": """You are the Air Quality and Pollution Specialist for EcoGPT. Assess severity from AQI, CO, NO2, VOC,
MQ2, MQ7, MQ135, PM. Identify sources: traffic (high NO2+CO), industrial (high VOC+MQ135), biomass burning (high MQ2+CO), or mixed.
Recommend short-term (24-72h), medium-term (1-6mo), long-term (1-5yr) actions. Specify bio-filter species per pollutant
(NO2: Ficus benjamina, Hedera helix; PM2.5: Tillandsia usneoides, Ficus elastica; VOC: Chlorophytum comosum, Spathiphyllum wallisii).
Quantify expected AQI improvement from green interventions. Flag heat-island risk if HI exceeds TMP by >3°C.""",
"plantation": """You are the Plantation and Biodiversity Specialist for EcoGPT. Recommend 8-12 named species with scientific
name, common name, suitability, ecological role, season, growth, CO2 rate, water need, native status. Design a plantation plan:
trees/ha scaled to population density, spatial arrangement, priority zones, 5-layer canopy strategy. Never recommend invasive
species (Prosopis juliflora, Lantana camara, Eichhornia crassipes). Always polyculture. Scale by density:
<500/km² large canopy restoration; 500-5000 urban park + avenue; >5000 vertical gardens, rooftop greening, compact flowering trees.""",
"water": """You are the Water Conservation Specialist for EcoGPT. Assess water stress from humidity, temperature, climate-zone
rainfall. Recommend water-body restoration (desilting, bunding, riparian buffers with Vetiver/Bamboo/Typha, lotus/lily bioremediation,
bioswales, fish stocking), rainwater harvesting with area calculations, check dams, retention ponds, irrigation efficiency.
Quantify litres saved per year and groundwater improvement estimates.""",
"urban": """You are the Urban Planning and Population Specialist for EcoGPT. Classify density (Rural <150, Peri-urban 150-1000,
Urban 1000-5000, Dense Urban >5000 /km²). Compute green-space deficit vs WHO 9 m²/capita. Recommend land-use optimization that
displaces no residents: institutional land, medians, buffers, riverbanks, rooftops; Miyawaki micro-forests for dense areas.
Recommend green corridors, permeable pavement, cool roofs, bio-retention cells. Estimate load reduction per intervention.""",
"carbon": """You are the Carbon Sequestration Estimation Specialist for EcoGPT. Use IPCC Tier-1 style factors and i-Tree
species data. Apply growth ramp (saplings ~10% of mature rate, full by year ~10) and 15% first-3-year mortality.
Project years 1/5/10/25. Compare against local emission load. Report tonnes CO2e/yr, cars-equivalent (4.6 t/car/yr),
households-equivalent. State confidence: High species-level / Medium genus / Low zone-average.""",
"synthesis": """You are the Synthesis Agent for EcoGPT. Merge all specialist JSON outputs, resolving contradictions by
deferring to the most data-grounded value. Output exactly: Environmental Assessment; Key Risks; Recommended Plant Species table;
Plantation Plan; Pollution Reduction Actions (short/medium/long); Soil Improvement; Water Conservation; Biodiversity Enhancement;
Carbon Sequestration table (yr 1/5/10/25); Predicted Environmental Impact; Priority Action Items ranked 1-10; What-If Simulation
with 2-3 scenarios. Calibrate tone to user type. Never fabricate species or figures; mark estimates "(estimated)" and assumptions "(assumed)".""",
}

# ---------------- Agent 1: Data Ingestion ----------------
def run_data_ingestion(lat, lon, radius=0.05, label=None):
    geo = reverse_geocode(lat, lon)
    # Prefer the user's own place name over reverse-geocoded suburb names
    # (e.g. 22.557,88.494 reverse-geocodes to "Newtown" — keep "Kolkata" if given)
    if label and not re.match(r"^\(", label):
        geo["city"] = label
    dens = get_population_density(lat, lon)
    local = load_sensor_data(lat, lon, radius)
    assumptions = []
    if len(local) < 10:  # widen search before giving up — nearby sensors still representative
        for r2 in (radius*3, radius*8):
            wider = load_sensor_data(lat, lon, r2)
            if len(wider) >= 10:
                local = wider
                assumptions.append(f"No readings within {radius}°; using {len(wider)} readings "
                                   f"from sensors within ~{r2*111:.0f} km (assumed representative)")
                break
    # Data priority: REAL uploaded IoT readings > LIVE per-location feeds >
    # synthetic demo rows > climate-zone model. Synthetic data never masks reality.
    if len(local) and "source" in local.columns:
        _real = local[~local["source"].astype(str).str.startswith("synthetic_")]
    else:
        _real = local
    use_real = len(_real) >= 10
    if use_real:
        local = _real
    live = fetch_live_environment(lat, lon)
    kz = geo["climate_zone"]
    use_sensors = use_real or (live is None and len(local) >= 10)
    if (not use_sensors) and live is not None:
        # REAL location-specific data — different for every place on Earth
        data_source = ("LIVE per-location feeds: Open-Meteo CAMS air quality + ERA5 weather "
                       "(real data for these exact coordinates)")
        assumptions.append("No IoT sensors here — live satellite/model data fetched for this exact location")
        assumptions.append("VOC/MQ channels not available from satellite feeds — "
                           "source apportionment limited to NO2/CO/PM (assumed)")
        stats = {"TMP":{"mean":live["tmp"]},"HMD":{"mean":live["hmd"]},"HI":{"mean":live["hi"]},
                 "CO":{"mean":live["co_ppm"]},"NO2":{"mean":live["no2_ppb"]/1000.0},
                 "VOC":{"mean":0.0},"RAWPM":{"mean":live["pm25"]},"DD":{"mean":live["pm10"]},
                 "MQ2":{"mean":0.0},"MQ7":{"mean":0.0},"MQ135":{"mean":0.0},"C2H5OH":{"mean":0.0}}
        completeness, anomalies = 0.85, []
    elif not use_sensors:
        data_source = "climate-zone model (offline fallback — all values estimated)"
        assumptions.append(f"No sensor coverage at ({lat:.3f},{lon:.3f}) and offline — "
                           "climate-zone modelled values used (estimated)")
        base = {"Aw":(31,65,8,0.005,40,120,200),"Af":(28,80,5,0.004,35,90,150),
                "BWh":(36,30,9,0.006,50,160,300),"BSh":(33,40,9,0.006,45,150,280),
                "Cfa":(22,65,4,0.004,30,60,90),"Cfb":(14,75,3,0.003,25,40,60),
                "Dfb":(10,70,3,0.003,25,45,70)}.get(kz,(25,60,5,0.004,35,80,120))
        t,h,co,no2,voc,pm,dd = base
        stats = {"TMP":{"mean":t},"HMD":{"mean":h},"CO":{"mean":co},"NO2":{"mean":no2},
                 "VOC":{"mean":voc},"RAWPM":{"mean":pm},"DD":{"mean":dd},
                 "HI":{"mean":heat_index_c(t,h)},"MQ2":{"mean":3},"MQ7":{"mean":3},
                 "MQ135":{"mean":5},"C2H5OH":{"mean":20}}
        completeness, anomalies = 0.0, []
    else:
        data_source = (f"local IoT sensor network ({len(local):,} real readings)" if use_real else
                       f"synthetic demo sensor data ({len(local):,} rows, calibrated to reference CSV) "
                       "— connect internet for live feeds or add real sensors")
        stats = {c: {"mean": float(local[c].mean()), "max": float(local[c].max()),
                     "min": float(local[c].min()), "std": float(local[c].std() or 0)}
                 for c in SENSOR_COLS if c in local}
        completeness = float(local[SENSOR_COLS].notna().mean().mean())
        anomalies = []
        for c in ["CO","NO2","VOC","RAWPM","DD","MQ2","MQ135"]:
            if c in stats and stats[c].get("std", 0) > 0:
                thresh = stats[c]["mean"] + 2*stats[c]["std"]
                n_anom = int((local[c] > thresh).sum())
                if n_anom: anomalies.append(f"{c}: {n_anom} readings >2σ (max {stats[c]['max']:.1f})")
        if local["RAWPM"].notna().mean() < 0.2:
            assumptions.append("RAWPM mostly missing — PM2.5 proxied from dust density ×0.4 (assumed)")
    pm_val = stats.get("RAWPM",{}).get("mean", float("nan"))
    if len(local) >= 10 and local.get("RAWPM") is not None and local["RAWPM"].notna().mean() < 0.2:
        pm_val = float("nan")
    aqi = compute_aqi(no2_ppm=stats.get("NO2",{}).get("mean"),
                      co_ppm=stats.get("CO",{}).get("mean"),
                      pm25=None if math.isnan(pm_val) else pm_val,
                      mq135=stats.get("MQ135",{}).get("mean"),
                      dd=stats.get("DD",{}).get("mean"))
    # optional Kaggle city-level baseline (Cell 2b): override when nothing better,
    # cross-check otherwise
    if "kaggle_city_baseline" in globals():
        kb_ = kaggle_city_baseline(geo["city"])
        if kb_ and not math.isnan(kb_.get("aqi", float("nan"))):
            if len(local) < 10 and live is None:
                aqi = {"score": round(kb_["aqi"],1), "category": aqi_category(kb_["aqi"]),
                       "dominant": "PM2.5", "sub_indices": {},
                       "notes": ["city-level Kaggle baseline used (static dataset)"]}
                data_source = "Kaggle global air-pollution city baseline (static)"
            else:
                aqi["notes"].append(f"Kaggle city baseline cross-check: AQI {kb_['aqi']:.0f}")
    solar = live["solar_kwh_m2_day"] if live else ZONE_SOLAR.get(kz, 4.5)
    wind = live["wind_ms"] if live else 3.5
    return {"location": {"city": geo["city"], "lat": lat, "lon": lon,
                         "country": geo["country"], "climate_zone": geo["climate_zone"],
                         "climate_name": CLIMATE_ZONE_NAMES.get(geo["climate_zone"], geo["climate_zone"])},
            "population_density": round(dens), "density_class": classify_density(dens),
            "aqi": aqi,
            "temperature_avg": round(stats["TMP"]["mean"],1), "humidity_avg": round(stats["HMD"]["mean"],1),
            "heat_index_avg": round(stats["HI"]["mean"],1),
            "co_avg": round(stats["CO"]["mean"],1), "no2_avg": round(stats["NO2"]["mean"],3),
            "voc_avg": round(stats["VOC"]["mean"],1),
            "pm_avg": round(stats.get("DD",{}).get("mean",0)*0.4,1),
            "data_source": data_source,
            "solar_kwh_m2_day": round(solar, 2), "wind_ms": round(wind, 1),
            "readings_used": len(local), "anomalies": anomalies,
            "data_completeness": round(completeness,2), "assumptions": assumptions,
            "stats": {k:{kk:round(vv,2) for kk,vv in v.items()} for k,v in stats.items()}}

# ---------------- Agent 3: Pollution ----------------
def run_pollution_agent(ctx):
    s = ctx["stats"]
    def g(c):
        v = s.get(c, {}).get("mean", 0) or 0
        return 0.0 if (isinstance(v, float) and math.isnan(v)) else v
    sigs = {"traffic": (g("NO2")*120 + g("CO")*0.6),
            "industrial": (g("VOC")*0.9 + g("MQ135")*4),
            "biomass": (g("MQ2")*8 + g("CO")*0.4)}
    top = max(sigs, key=sigs.get)
    spread = sorted(sigs.values(), reverse=True)
    source = top if spread[0] > spread[1]*1.25 else "mixed"
    heat_island = ctx["heat_index_avg"] - ctx["temperature_avg"] > 3
    chunks = retrieve(build_retrieval_query("pollution", ctx), 3)
    cat = ctx["aqi"]["category"]
    score = ctx["aqi"]["score"]
    # ---- actions assembled from severity + detected source mix (not static lists) ----
    hedge_sp = filter_species_by_climate(ctx["location"]["climate_zone"], ctx["density_class"],
                                         3, country=ctx["location"].get("country",""))
    hedge = ", ".join(s["common"] for s in hedge_sp)
    short, medium, longt = [], [], []
    if score > 150:
        short += ["Public health advisory: N95 masks, limit outdoor exertion at peak hours",
                  "Halt construction & demolition 72h; water-sprinkle arterial roads twice daily"]
    elif score > 100:
        short += ["Sensitive-group advisory (children, elderly, cardio-respiratory patients)",
                  "Mechanised sweeping + water sprinkling on dust hotspots"]
    else:
        short += ["Maintain monitoring; publish daily AQI bulletins to sustain good air"]
    if source in ("traffic", "mixed"):
        short += ["Traffic demand management: odd-even / heavy-vehicle daytime restrictions, "
                  "public-transport fare incentives"]
        medium += [f"3-row green barrier hedges ({hedge}) along the worst arterials",
                   "Junction redesign + idling-engine enforcement at choke points"]
        longt += ["EV transition zone: e-bus depots, charging mandates in new buildings"]
    if source in ("industrial", "mixed"):
        short += ["Spot-check stack emissions at red-category units; suspend violators"]
        medium += ["Continuous emission monitoring on major stacks; VOC capture at solvent users"]
        longt += ["500 m industrial buffer greenbelts; relocate non-compliant units"]
    if source in ("biomass", "mixed"):
        short += ["Anti-open-burning patrols; fine refuse/leaf burning"]
        medium += ["LPG/electric conversion for street-food and small eateries; "
                   "decentralised composting so green waste is never burnt"]
    medium += ["Filtered ventilation retrofits in schools and clinics in hotspot zones"]
    longt += ["Urban forestry corridors connecting parks at ≤500 m spacing (3-30-300 rule)",
              "Green building code: cool roofs (albedo >0.65) + rooftop gardens"]
    if heat_island:
        longt += ["Heat-island programme: 30% canopy target + cool-roof retrofits on public buildings"]
    biofilters = {"NO2": ["Ficus benjamina","Hedera helix (green screens)","Azadirachta indica"],
                  "PM2.5/dust": ["Ficus elastica","Tillandsia usneoides","Neem hedgerows","Betula pendula"],
                  "VOC": ["Chlorophytum comosum","Spathiphyllum wallisii","Epipremnum aureum"]}
    return {"severity": cat, "aqi_score": ctx["aqi"]["score"], "dominant_pollutant": ctx["aqi"]["dominant"],
            "source_apportionment": {"classification": source,
                "signals": {k: round(v,1) for k,v in sigs.items()}},
            "heat_island_flag": heat_island,
            "heat_island_delta": round(ctx["heat_index_avg"]-ctx["temperature_avg"],1),
            "actions": {"short_term_24_72h": short, "medium_term_1_6mo": medium,
                        "long_term_1_5yr": longt},
            "biofilter_species": biofilters,
            "expected_improvement": "Street-canyon green walls: up to 40% NO2 / 60% PM locally; "
                "citywide canopy at 30%: 2-10% PM2.5 reduction + 1-3°C cooling (Pugh et al., estimated)",
            "grounding": [{"id":c["id"],"source":c["source"],"relevance":c["relevance"]} for c in chunks]}

# ---------------- Agent 4: Plantation & Biodiversity ----------------
def run_plantation_agent(ctx):
    dens_class = ctx["density_class"]
    species = filter_species_by_climate(ctx["location"]["climate_zone"], dens_class, 12,
                                        country=ctx["location"].get("country", ""))
    trees_ha = {"Rural": 400, "Peri-urban": 250, "Urban": 150, "Dense Urban": 100}[dens_class]
    arrangement = {"Rural": "Block restoration + farm bunds (agroforestry rows at 10 m)",
        "Peri-urban": "Cluster planting in commons + avenue rows on connecting roads",
        "Urban": "Avenue planting (8-10 m spacing) + pocket parks + biodiversity corridors",
        "Dense Urban": "Miyawaki micro-forests (3 saplings/m² on ≥30 m² plots), vertical gardens, "
                       "rooftop greening, compact flowering avenues"}[dens_class]
    chunks = retrieve(build_retrieval_query("plantation", ctx), 4)
    # ---- canopy layers derived from the ACTUAL species selected for this location ----
    layers = {}
    for s in species:
        layers.setdefault(s["layer"], []).append(s["scientific"])
    canopy_layers = {lyr: ", ".join(names[:3]) for lyr, names in layers.items()}
    for lyr in ("emergent","canopy","sub-canopy","shrub","ground"):
        canopy_layers.setdefault(lyr, "— (extend species DB for this layer in this climate zone)")
    priority_zones = {
        "Rural": ["Degraded commons & village forest blocks","Farm bunds (agroforestry rows)",
                  "Riverbanks & pond margins","School and panchayat/communal grounds"],
        "Peri-urban": ["Connecting-road avenues","Commons & grazing-land edges",
                       "Water-body bunds","Institutional campuses","Peri-urban wetland buffers"],
        "Urban": ["Roadsides & medians","Water-body bunds & riparian belts",
                  "Institutional campuses (schools/hospitals/depots)","Pocket parks on vacant lots"],
        "Dense Urban": ["Vacant municipal lots → Miyawaki plots","Rooftops & flyover pillars",
                        "Roadside verges & medians","Institutional campuses","Canal banks"],
    }[dens_class]
    return {"species": species, "trees_per_hectare": trees_ha,
            "spatial_arrangement": arrangement,
            "priority_zones": priority_zones,
            "canopy_layers": canopy_layers,
            "avoid": INVASIVE_BLACKLIST,
            "polyculture_rule": "No species >15% of total; ≥10 species/ha",
            "grounding": [{"id":c["id"],"source":c["source"],"relevance":c["relevance"]} for c in chunks]}

# ---------------- Agent 5: Water ----------------
REGION_FISH = {"India": "Rohu, Catla, Mrigal (Indian major carps)",
               "East Africa": "native Oreochromis/Labeo species (consult fisheries dept)",
               "Europe": "native cyprinids (roach, rudd, tench) — agency approval required"}

def run_water_agent(ctx):
    h, t = ctx["humidity_avg"], ctx["temperature_avg"]
    L = ctx["location"]
    rain = get_annual_rainfall(L["lat"], L["lon"])
    stress = ("High (dry + hot: evaporation stress)" if h < 40 and t > 35 else
              "High (arid climate)" if h < 40 else
              "Moderate" if h <= 70 else "Low (humid — drainage & waterlogging priority)")
    region = COUNTRY_TO_REGION.get((L.get("country") or "").strip().lower())
    fish = REGION_FISH.get(region, "native fish per local fisheries authority (never exotics)")
    # riparian palette from the species DB for THIS climate zone
    rip = [s["scientific"] for s in filter_species_by_climate(
        L["climate_zone"], "Rural", 12, country=L.get("country","")) if s["water"] in ("high","aquatic")]
    rip = ", ".join(rip[:3]) if rip else "deep-rooted native riparian species"
    roof, persons = 100, 5
    harvest_l = roof * rain * 0.8
    chunks = retrieve(build_retrieval_query("water", ctx), 3)
    plan = ["Intercept/divert sewage inflows BEFORE physical works",
            "Dry-season desilting to original bed; reuse tested silt on bunds",
            f"Vetiver hedgerows on bunds; 10-30 m riparian buffer ({rip})",
            "Reed beds (Typha/Phragmites) at inlets; floating-leaf natives ≤30% surface",
            "Bioswales + constructed wetland for stormwater first-flush",
            f"Restock {fish} once dissolved oxygen >4 mg/L"]
    if L["climate_zone"].startswith("A") or region == "India":
        plan.insert(4, "Remove Eichhornia (water hyacinth) fully — weevil biocontrol + composting")
    if stress.startswith("High"):
        plan = ["PRIORITY: check dams + percolation ponds for groundwater recharge",
                "Mulch + drip for all new plantations (no flood irrigation)"] + plan
    elif stress.startswith("Low"):
        plan = ["PRIORITY: drainage management — permeable paving, retention ponds, "
                "wetland buffers against waterlogging/flood"] + plan
    return {"stress_level": stress, "annual_rainfall_mm": rain,
        "restoration_plan": plan,
        "rainwater_harvesting": {
            "per_100m2_roof_litres_yr": int(harvest_l),
            "household_demand_coverage_pct": round(100*harvest_l/(persons*135*365),1),
            "recharge_structures": "1-2 m³ percolation pit per 100 m² roof; boundary trenches; defunct-borewell shafts",
            "groundwater_response": "0.3-1.0 m table rise in 3-5 monsoons at ward scale (CGWB pilots, estimated)"},
        "irrigation": ["Flood→drip conversion saves ~3-4 ML/ha/yr (90% vs 40% efficiency)",
                       "Soil-moisture-sensor scheduling: further 15-25% saving"],
        "quantified_savings_l_yr": {"rwh_per_1000_households": int(harvest_l*1000),
                                    "drip_per_ha": 3_500_000},
        "grounding": [{"id":c["id"],"source":c["source"],"relevance":c["relevance"]} for c in chunks]}

# ---------------- Agent 6: Urban Planning ----------------
def run_urban_agent(ctx):
    d = ctx["population_density"]; dc = ctx["density_class"]
    deficit_m2_km2 = d * 9                      # WHO 9 m²/capita per km² of city
    green_needed_ha_km2 = deficit_m2_km2 / 10_000
    chunks = retrieve(build_retrieval_query("urban", ctx), 3)
    city = ctx["location"]["city"].lower()
    note = ("Current green cover should be measured via Sentinel-2/MODIS NDVI "
            "(see satellite layer in the dashboard, Cell 11)")
    if "kolkata" in city:
        note += "; Kolkata reference ≈2 m²/capita vs WHO 9 → ~7 m²/person deficit (estimated)"
    co2_pc = {"india":2.0,"bangladesh":0.7,"kenya":0.4,"nigeria":0.6,"united kingdom":4.7,
              "germany":7.9,"united states":14.7,"uae":21.8,"japan":8.5,"brazil":2.2,
              "singapore":8.9,"egypt":2.5,"russia":11.4,"australia":15.0
              }.get((ctx["location"].get("country") or "").lower(), 4.7)
    trees_per_1000 = int(1000 * co2_pc * 1000 / 22 / 1000)  # trees to offset 1% would be /100
    return {"density_per_km2": d, "classification": dc,
        "who_green_norm_m2pc": 9,
        "required_green_ha_per_km2": round(green_needed_ha_km2, 1),
        "per_capita_co2_t": co2_pc,
        "trees_per_1000_residents_full_offset": trees_per_1000,
        "note": note,
        "greening_without_displacement": ["Institutional campuses (15-30% of urban land)",
            "Road medians & verges (avenue planting)","Canal/river banks","Industrial buffers",
            "Rooftops (10-20% of plan area feasible)","Parking lots (40% shade-tree mandate)"],
        "dense_urban_toolkit": ["Miyawaki micro-forests: ≥90 trees per 30 m² plot",
            "Vertical gardens on flyover pillars (1 m² ≈ 2.3 kg CO2/yr + PM capture)",
            "Cool roofs (albedo >0.65) + 30% canopy target → 1-3°C ambient cooling",
            "Permeable pavement in markets (70-90% runoff cut)",
            "Bio-retention cells every 300 m of storm drain"],
        "co2_honesty_note": f"At ~{co2_pc} t CO2/person/yr here, ~{max(1,int(co2_pc*1000/22/100))*10} "
            "mature trees per 1000 residents offset only ~1% of their emissions — urban forests are "
            "for air, heat & habitat; pair with energy transition (estimated)",
        "grounding": [{"id":c["id"],"source":c["source"],"relevance":c["relevance"]} for c in chunks]}

# ---------------- Agent 7: Carbon ----------------
def run_carbon_agent(ctx, plantation, trees_to_plant=None):
    sp = plantation["species"][:10]
    if trees_to_plant is None:
        trees_to_plant = {"Rural": 50000, "Peri-urban": 25000,
                          "Urban": 15000, "Dense Urban": 10000}[ctx["density_class"]]
    mean_rate = float(np.mean([s["co2_kg_yr"] for s in sp]))  # kg/tree/yr mature
    def cum(years):
        tot = 0.0
        for y in range(1, years+1):
            ramp = min(0.10 + 0.10*y, 1.0)         # logistic-ish growth ramp
            surv = 0.85 if y >= 3 else (1 - 0.05*y)  # 15% mortality over first 3 yrs
            tot += trees_to_plant * mean_rate * ramp * surv
        return tot/1000.0                           # tonnes
    proj = {f"year_{y}": round(cum(y),1) for y in (1,5,10,25)}
    annual_at_maturity = trees_to_plant * mean_rate * 0.85 / 1000
    chunks = retrieve(build_retrieval_query("carbon", ctx), 2)
    return {"trees_modelled": trees_to_plant,
        "mean_species_rate_kg_yr": round(mean_rate,1),
        "cumulative_tonnes_co2e": proj,
        "annual_tonnes_at_maturity": round(annual_at_maturity,1),
        "cars_equivalent_at_maturity": int(annual_at_maturity/4.6),
        "households_equivalent": int(annual_at_maturity/1.5),
        "mortality_assumption": "15% cumulative in first 3 years (assumed)",
        "growth_curve": "saplings ~10% of mature uptake yr-1, full by ~yr-9 (IPCC Tier-1 style)",
        "confidence": "High (species-level i-Tree rates for "
                      f"{len(sp)} recommended species)",
        "grounding": [{"id":c["id"],"source":c["source"],"relevance":c["relevance"]} for c in chunks]}

# ---------------- Soil (folded into Water & Soil per architecture) ----------------
def run_soil_recommendations(ctx):
    kz = ctx["location"]["climate_zone"]
    chunks = retrieve(build_retrieval_query("soil", ctx), 2)
    recs = ["Compost 5-10 t/ha/yr; vermicompost for urban beds",
            "Biochar 2-5 t/ha (+15-25% water holding)",
            "Planting pits 1×1×1 m: 60% soil + 30% compost + 10% sand, mycorrhizal inoculation"]
    if kz.startswith("A"):       # tropical
        recs = ["Green manure (Sesbania 45-day cycle: +60-80 kg N/ha)"] + recs + \
               ["Mulching: 30-50% less surface evaporation; no bare soil in monsoon"]
    elif kz in ("BWh","BSh"):    # arid
        recs += ["Drip-basin micro-catchments around every sapling; gypsum for sodic patches",
                 "Thick mulch is critical: 50%+ evaporation cut in arid heat"]
    else:                        # temperate/continental
        recs += ["Leaf-mould composting of autumn litter; avoid winter soil compaction",
                 "Street-tree pits ≥6 m³ with structural soil under pavements",
                 "Cover crops (clover/vetch) on any soil bare over winter"]
    if ctx["location"]["city"].lower() in ("kolkata","sundarbans"):
        recs.append("Coastal fringe salinity: prefer Casuarina, Pongamia; test EC before food crops")
    return {"recommendations": recs,
            "grounding": [{"id":c["id"],"source":c["source"],"relevance":c["relevance"]} for c in chunks]}

# ---------------- Agent 9: Renewable Energy & Interventions (feasibility-checked) ----------------
GRID_EF = {"india":0.71,"bangladesh":0.67,"kenya":0.10,"nigeria":0.52,"united kingdom":0.21,
           "germany":0.38,"france":0.06,"united states":0.37,"uae":0.49,"japan":0.46,
           "brazil":0.10,"singapore":0.41,"egypt":0.47,"russia":0.36,"australia":0.66}

def run_energy_agent(ctx):
    """Every intervention is gated on SITE FEASIBILITY: terrain slope (real
    elevation data), land availability by density class, measured insolation,
    wind speed, rainfall. Never proposes clearing green cover or building on
    steep/mountainous ground."""
    L = ctx["location"]
    terrain = get_terrain(L["lat"], L["lon"]) or {
        "elevation_m": None, "relief_m_per_km": None,
        "slope_class": "unknown (offline — verify slope on site before any ground works)"}
    solar, wind = ctx.get("solar_kwh_m2_day", 4.5), ctx.get("wind_ms", 3.5)
    dens, dc = ctx["population_density"], ctx["density_class"]
    grid_ef = GRID_EF.get((L.get("country") or "").lower(), 0.44)
    kwh_kwp = solar * 0.75 * 365                       # per kWp/yr at PR 0.75
    rain = get_annual_rainfall(L["lat"], L["lon"])
    steep = terrain["slope_class"] in ("steep", "mountainous")
    items = []
    add = lambda n, f, r, p="": items.append(
        {"intervention": n, "feasible": f, "reason": r, "potential": p})

    add("Rooftop solar PV",
        "✅ Yes" if solar >= 3.5 else "⚠️ Conditional" if solar >= 2.2 else "❌ No",
        f"insolation {solar:.1f} kWh/m²/day; uses existing roofs — no land/terrain constraint",
        f"1 kWp ≈ {kwh_kwp:,.0f} kWh/yr ≈ {kwh_kwp*grid_ef/1000:.2f} t CO2/yr avoided")
    if steep:
        add("Ground-mounted solar farm", "❌ No",
            f"terrain is {terrain['slope_class']} (relief {terrain['relief_m_per_km']} m/km) — "
            "grading risks erosion/landslides; rooftop & canopy solar only")
    elif dc in ("Dense Urban", "Urban"):
        add("Ground-mounted solar farm", "⚠️ Conditional",
            "open urban land is scarce and better used for green cover — restrict to parking "
            "canopies, canal-top arrays and capped landfills",
            "canal-top: ~1 MWp per km of 10 m-wide canal + evaporation savings")
    else:
        add("Ground-mounted solar farm", "✅ Yes" if solar >= 4 else "⚠️ Conditional",
            f"{terrain['slope_class']} terrain, {dc.lower()} land — ONLY degraded/barren plots; "
            "NEVER clear vegetation for panels (carbon payback turns negative)",
            f"1 ha ≈ 0.8 MWp ≈ {0.8*kwh_kwp:,.0f} MWh/yr")
    add("Agrivoltaics (panels over crops)",
        "❌ No" if steep else "✅ Yes" if dc in ("Rural","Peri-urban") and solar >= 4 else "⚠️ Conditional",
        "terrain too steep for arrays" if steep else
        "dual land use: farmland stays productive, panels cut crop heat stress 1-3°C")
    add("Small wind turbines",
        "✅ Yes" if wind >= 5.5 else "⚠️ Conditional" if wind >= 4.0 else "❌ No",
        f"mean wind {wind:.1f} m/s here (viable ≥4, good ≥5.5)")
    if (terrain["relief_m_per_km"] or 0) >= 60 and rain >= 1000:
        add("Micro-hydro", "⚠️ Conditional",
            f"relief {terrain['relief_m_per_km']} m/km + {rain} mm rain/yr — "
            "survey perennial streams for head & flow before committing")
    else:
        add("Micro-hydro", "❌ No",
            f"insufficient slope ({terrain['relief_m_per_km'] or 'unknown'} m/km) "
            f"and/or rainfall ({rain} mm) for usable head/flow")
    add("Community biogas (organic waste)",
        "✅ Yes" if dens > 1000 else "⚠️ Conditional",
        f"{dc}: ≈{int(dens*0.35):,} kg organic waste/km²/day (0.35 kg/person) feeds digesters",
        "40-60 m³ biogas per tonne wet waste; digestate fertilises the plantations")
    if dc == "Rural":
        add("Solar irrigation pumps", "✅ Yes" if solar >= 4 else "⚠️ Conditional",
            f"replaces diesel pumps; insolation {solar:.1f} kWh/m²/day",
            "each 5 HP pump ≈ 2-3 t CO2/yr avoided")
    else:
        add("EV charging network", "✅ Yes",
            "pairs with the EV-transition zone in pollution actions; feed from rooftop solar")
    if ctx["heat_index_avg"] - ctx["temperature_avg"] > 3:
        add("Cool roofs (albedo >0.65)", "✅ Yes",
            f"heat island (+{ctx['heat_index_avg']-ctx['temperature_avg']:.1f}°C feels-like) — "
            "roof surface −25°C, AC load −15-30%")
    chunks = retrieve("renewable energy siting feasibility solar slope terrain wind biogas", 2)
    return {"terrain": terrain, "solar_kwh_m2_day": round(solar,2), "wind_ms": round(wind,1),
            "grid_emission_factor_kg_kwh": grid_ef, "interventions": items,
            "grounding": [{"id":c["id"],"source":c["source"],"relevance":c["relevance"]} for c in chunks]}

print("Agents defined ✓  (data, rag, pollution, plantation, water, urban, carbon, soil, energy + synthesis next)")


In [ ]:
# ============================================================
# Cell 9: Synthesis Agent + Orchestrator
#   Deterministic pipeline always runs; LLM (Ollama/HF) polishes
#   the final report and answers free-form follow-ups with RAG.
# ============================================================

_LOC_NOISE = {"location","improve","environment","focus","water","tree","trees","pollution",
              "population","density","analysis","report","high","given","bodies","planting",
              "reduction","the","and","here","how","what","which","restoration","riverbank"}

def _location_candidates(query: str) -> list:
    """Possible place-name phrases, most explicit first."""
    cands = []
    m = re.search(r"location\s*[:\-]\s*([^\n(.;]+)", query, re.I)
    if m: cands.append(m.group(1))
    for m in re.finditer(r"\b(?:in|at|near|around|for)\s+([A-Z][\w'\-]+(?:[ ,]+[A-Z][\w'\-]+){0,3})", query):
        cands.append(m.group(1))
    q = query.strip()
    if len(q) <= 45 and "?" not in q:        # bare input like "Paris" / "Cape Town, SA"
        cands.append(q)
    for m in re.finditer(r"\b([A-Z][a-zA-Z'\-]{2,}(?:\s+[A-Z][a-zA-Z'\-]{2,}){0,3})\b", query):
        cands.append(m.group(1))
    seen, out = set(), []
    for c in cands:
        c = c.strip(" ,.;:-")
        if not c or c.lower() in seen: continue
        words = [w for w in re.findall(r"[A-Za-z'\-]+", c) if w.lower() not in _LOC_NOISE]
        if not words: continue
        seen.add(c.lower()); out.append(" ".join(words) if len(words) < len(c.split()) else c)
    return out

def parse_location(query: str):
    """Extract location → (lat, lon, label) or None. NEVER silently defaults:
    coords → gazetteer → Nominatim (any place on Earth)."""
    m = (re.search(r"lat[:\s]*(-?\d+\.?\d*)[,\s]+lon[g]?[:\s]*(-?\d+\.?\d*)", query, re.I)
         or re.search(r"\(?\s*(-?\d{1,2}\.\d+)\s*,\s*(-?\d{1,3}\.\d+)\s*\)?", query))
    if m:
        lat, lon = float(m.group(1)), float(m.group(2))
        return lat, lon, f"({lat:.3f}, {lon:.3f})"
    ql = query.lower()
    for name in GAZETTEER:
        if re.search(rf"\b{re.escape(name)}\b", ql):
            return GAZETTEER[name][0], GAZETTEER[name][1], name.title()
    for cand in _location_candidates(query):
        key = cand.lower()
        for name in GAZETTEER:
            if name in key:
                return GAZETTEER[name][0], GAZETTEER[name][1], name.title()
        if CAPS["geopy"]:
            try:
                loc = Nominatim(user_agent="ecogpt_hackathon", timeout=6).geocode(cand)
                if loc:
                    return loc.latitude, loc.longitude, cand
            except Exception:
                continue
    return None

LOCATION_HELP = ("❓ **I couldn't identify a location in your message.** Please give me a city "
    "(e.g. `Nairobi`, `Location: Pune, India`) or coordinates (`22.557, 88.494`). "
    "Online geocoding covers any place on Earth; offline I know: "
    + ", ".join(sorted(n.title() for n in GAZETTEER)) + ".")

def run_synthesis(ctx, pol, plant, water, urban, carbon, soil, energy, user_type="default"):
    L = ctx["location"]
    sp_rows = "\n".join(
        f"| {s['scientific']} | {s['common']} | {s['role'][:45]} | {s['co2_kg_yr']} | {s['water']} | {s['native_to']} |"
        for s in plant["species"])
    pr = carbon["cumulative_tonnes_co2e"]
    # scenarios + assumed settlement area scale with the density class (not Kolkata constants)
    base_n = {"Rural": 50000, "Peri-urban": 25000, "Urban": 15000, "Dense Urban": 10000}[ctx["density_class"]]
    area_ha = {"Rural": 150000, "Peri-urban": 60000, "Urban": 35000, "Dense Urban": 20000}[ctx["density_class"]]
    scenario_set = [(base_n, 5), (base_n*3, 10), (base_n*6, 25)]
    sims = [simulate_intervention(ctx["aqi"]["score"], n, n/plant["trees_per_hectare"], y,
                                  urban_area_ha=area_ha) for n, y in scenario_set]
    sim_txt = "\n".join(
        f"- **If {n:,} trees are planted over {y} years:** AQI {ctx['aqi']['score']} → "
        f"{s['projected_aqi']} (−{s['aqi_reduction']}), {s['co2_captured_tonnes']:,} t CO2e captured "
        f"(≈{s['cars_equivalent']:,} cars/yr), +{s['green_cover_increase_pct']}% green cover *(medium confidence)*"
        for (n, y), s in zip(scenario_set, sims))
    # biodiversity narrative from the ACTUAL species mix + climate zone
    kz = L["climate_zone"]
    bloom = [f"{s['common']} ({s['season']})" for s in plant["species"][:6]]
    keystone = next((s for s in plant["species"] if "keystone" in s["role"].lower()
                     or "Ficus" in s["scientific"] or "Quercus" in s["scientific"]), None)
    bio_lines = [
        f"- 5-layer canopy from the selected mix (see Plantation Plan); planting windows: {', '.join(bloom)}",
        (f"- Keystone species: *{keystone['scientific']}* ({keystone['common']}) — anchor for birds & pollinators"
         if keystone else "- Add a keystone fig/oak equivalent for this zone to anchor vertebrate food webs"),
        "- Stepping-stone pocket parks every 500 m; dead-wood retention; no-mow meadow patches",
        ("- Wetland margins <30 cm deep + reed beds for waterbirds" if kz.startswith("A") else
         "- Flowering desert natives + shaded water points for arid-zone pollinators and birds"
         if kz.startswith("B") else
         "- Hedgerow connectivity + nectar strips for temperate pollinators")]
    anomalies = "\n".join(f"- {a}" for a in ctx["anomalies"]) or "- None flagged"
    # ---------- plain-language "At a Glance" with letter grades ----------
    def _grade(val, bands):  # bands = [(ceiling, grade), …]
        for ceil_, gr in bands:
            if val <= ceil_: return gr
        return "F"
    g_air = _grade(ctx["aqi"]["score"], [(50,"A"),(100,"B"),(150,"C"),(200,"D"),(300,"E")])
    g_heat = _grade(pol["heat_island_delta"], [(1,"A"),(3,"B"),(5,"C"),(8,"D")])
    g_water = {"L":"B","M":"B","H":"D"}.get(water["stress_level"][0], "C")
    sp3 = ", ".join(s["common"] for s in plant["species"][:3])
    y10 = carbon["cumulative_tonnes_co2e"]["year_10"]
    at_a_glance = f"""## 📌 At a Glance (easy summary)
| | Status | Grade |
|---|---|---|
| 🌬 Air | AQI **{ctx['aqi']['score']} – {ctx['aqi']['category']}**, mainly from **{pol['source_apportionment']['classification']}** sources | **{g_air}** |
| 🔥 Heat | Feels **{pol['heat_island_delta']}°C hotter** than actual ({"heat-island problem" if pol['heat_island_flag'] else "acceptable"}) | **{g_heat}** |
| 💧 Water | Stress: **{water['stress_level'].split('(')[0].strip()}** · rain ≈{water['annual_rainfall_mm']} mm/yr | **{g_water}** |
| 🌳 Green | **{ctx['density_class']}** ({ctx['population_density']:,}/km²) → needs ~**{urban['required_green_ha_per_km2']} ha/km²** more green | **{"D" if ctx['density_class']=="Dense Urban" else "C" if ctx['density_class']=="Urban" else "B"}** |
| ⚡ Energy | Solar **{energy['solar_kwh_m2_day']} kWh/m²/day** · wind {energy['wind_ms']} m/s · terrain **{energy['terrain']['slope_class']}** | **{_grade(-energy['solar_kwh_m2_day'], [(-5.5,"A"),(-4.5,"B"),(-3.5,"C"),(-2.5,"D")])}** |

**In one paragraph:** {L['city']}'s air is *{ctx['aqi']['category'].lower()}* and the biggest quick win is tackling {pol['source_apportionment']['classification']} emissions. Plant **{carbon['trees_modelled']:,} trees** (start with {sp3}) on the open land shown in the dashboard — in 10 years that captures ≈**{y10:,.0f} t of CO2** (like taking {int(y10/4.6/10):,} cars off the road each year). Harvest rooftop rain (**{water['rainwater_harvesting']['per_100m2_roof_litres_yr']:,} L/yr per 100 m² roof**) and fix water bodies starting with: *{water['restoration_plan'][0].lower()}*.
"""
    all_assumptions = ctx["assumptions"] + ctx["aqi"].get("notes", [])
    assumptions = "; ".join(all_assumptions) if all_assumptions else "none"
    report = f"""
# 🌿 EcoGPT Environmental Report — {L['city']}, {L['country']}

{at_a_glance}

## Environmental Assessment
**Location:** {L['city']} ({L['lat']:.3f}, {L['lon']:.3f}) | **Climate:** {L['climate_name']} ({L['climate_zone']})
**AQI:** {ctx['aqi']['score']} ({ctx['aqi']['category']}) — dominant pollutant: {ctx['aqi']['dominant']}
**Temp:** {ctx['temperature_avg']}°C | **Humidity:** {ctx['humidity_avg']}% | **Heat Index:** {ctx['heat_index_avg']}°C
**Population density:** ~{ctx['population_density']:,}/km² ({ctx['density_class']})
**Data source:** {ctx['data_source']} | **Readings:** {ctx['readings_used']:,} (completeness {ctx['data_completeness']:.0%})
**Assumptions:** {assumptions}

## Key Risks Identified
- Pollution severity **{pol['severity']}**; likely source mix: **{pol['source_apportionment']['classification']}** (signals {pol['source_apportionment']['signals']})
- {"⚠️ **Heat island**: heat index exceeds ambient by " + str(pol['heat_island_delta']) + "°C" if pol['heat_island_flag'] else "Heat island risk currently low (HI−TMP = " + str(pol['heat_island_delta']) + "°C)"}
- Water stress: **{water['stress_level']}** (annual rainfall ≈{water['annual_rainfall_mm']} mm)
- Green-space requirement: **{urban['required_green_ha_per_km2']} ha/km²** to meet WHO 9 m²/capita ({urban['note']})
- Sensor anomalies:
{anomalies}

## Recommended Plant Species
| Scientific name | Common | Role | CO2 (kg/yr) | Water | Status |
|---|---|---|---|---|---|
{sp_rows}

*Avoid (invasive):* {", ".join(plant['avoid'])}. *Rule:* {plant['polyculture_rule']}.

## Plantation Plan
- **Density:** {plant['trees_per_hectare']} trees/ha ({ctx['density_class']} scaling)
- **Arrangement:** {plant['spatial_arrangement']}
- **Priority zones:** {"; ".join(plant['priority_zones'])}
- **Canopy layers (from selected species):** {"; ".join(f"{lyr} — {names}" for lyr, names in plant['canopy_layers'].items() if not names.startswith("—"))}

## Pollution Reduction Actions
**Short-term (24–72 h):** {"; ".join(pol['actions']['short_term_24_72h'])}
**Medium-term (1–6 mo):** {"; ".join(pol['actions']['medium_term_1_6mo'])}
**Long-term (1–5 yr):** {"; ".join(pol['actions']['long_term_1_5yr'])}
**Bio-filters:** NO2 → {", ".join(pol['biofilter_species']['NO2'])} | PM → {", ".join(pol['biofilter_species']['PM2.5/dust'])} | VOC → {", ".join(pol['biofilter_species']['VOC'])}
**Expected gain:** {pol['expected_improvement']}

## Soil Improvement Actions
{chr(10).join("- " + r for r in soil['recommendations'])}

## Water Conservation Recommendations
{chr(10).join("- " + r for r in water['restoration_plan'])}
- **RWH:** {water['rainwater_harvesting']['per_100m2_roof_litres_yr']:,} L/yr per 100 m² roof ({water['rainwater_harvesting']['household_demand_coverage_pct']}% of a 5-person household demand); {water['rainwater_harvesting']['recharge_structures']}
- **Groundwater:** {water['rainwater_harvesting']['groundwater_response']}
- **Irrigation:** {"; ".join(water['irrigation'])}

## Biodiversity Enhancement Plan
{chr(10).join(bio_lines)}

## Carbon Sequestration Estimate ({carbon['trees_modelled']:,} trees, mean {carbon['mean_species_rate_kg_yr']} kg CO2/tree/yr)
| Year 1 | Year 5 | Year 10 | Year 25 |
|---|---|---|---|
| {pr['year_1']:,} t | {pr['year_5']:,} t | {pr['year_10']:,} t | {pr['year_25']:,} t |

At maturity: **{carbon['annual_tonnes_at_maturity']:,} t CO2e/yr** ≈ {carbon['cars_equivalent_at_maturity']:,} cars removed ≈ {carbon['households_equivalent']:,} households' electricity. {carbon['mortality_assumption']}; {carbon['growth_curve']}. Confidence: {carbon['confidence']}.
{urban['co2_honesty_note']} *(estimated)*

## Renewable Energy & Other Improvement Scopes (feasibility-checked for this site)
**Site:** terrain **{energy['terrain']['slope_class']}**{f" (elev {energy['terrain']['elevation_m']} m, relief {energy['terrain']['relief_m_per_km']} m/km)" if energy['terrain']['elevation_m'] is not None else ""} | **Solar:** {energy['solar_kwh_m2_day']} kWh/m²/day | **Wind:** {energy['wind_ms']} m/s | **Grid:** {energy['grid_emission_factor_kg_kwh']} kg CO2/kWh

| Intervention | Feasible here? | Site-specific reason | Potential |
|---|---|---|---|
{chr(10).join(f"| {i['intervention']} | {i['feasible']} | {i['reason']} | {i['potential']} |" for i in energy['interventions'])}

## Predicted Environmental Impact
With full implementation over 5 years: AQI improvement {"15–30" if ctx['aqi']['score'] > 150 else "5–15"} points locally *(estimated)*{", ambient cooling 1–3°C in greened zones" if pol['heat_island_flag'] else ""}, runoff reduction 50–80% on treated catchments, measurable pollinator and bird recovery within 3 years.

## Priority Action Items
{chr(10).join(f"{i}. {item}" for i, item in enumerate([
    water['restoration_plan'][0],
    f"Launch {plant['species'][0]['season']} plantation: {plant['trees_per_hectare']} trees/ha mixed natives in: {plant['priority_zones'][0]}",
    pol['actions']['short_term_24_72h'][0],
    pol['actions']['medium_term_1_6mo'][0],
    urban['dense_urban_toolkit'][0] if ctx['density_class'] == 'Dense Urban' else urban['greening_without_displacement'][0] + ' greening drive',
    f"Rainwater harvesting: {water['rainwater_harvesting']['per_100m2_roof_litres_yr']:,} L/yr per 100 m² roof — mandate + retrofit programme",
    soil['recommendations'][0],
    pol['actions']['long_term_1_5yr'][0],
    f"Biodiversity: {bio_lines[1].lstrip('- ')}",
    "Quarterly sensor-network review vs this baseline (re-run EcoGPT)",
], 1))}

## What-If Simulation
{sim_txt}

---
*Grounded in: {", ".join(sorted(set(g['source'] for agent in (pol,plant,water,urban,carbon,soil,energy) for g in agent['grounding'])))}.*
*Figures marked (estimated)/(assumed) follow the EcoGPT no-fabrication policy.*
"""
    return report.strip()

def ask_ecogpt(query: str, user_type: str = "default", polish: bool = True) -> str:
    """Full EcoGPT pipeline: location → ingestion → specialists → synthesis → (LLM polish)."""
    loc = parse_location(query)
    if loc is None:
        return LOCATION_HELP
    lat, lon, label = loc
    ctx    = run_data_ingestion(lat, lon, label=label)
    pol    = run_pollution_agent(ctx)
    plant  = run_plantation_agent(ctx)
    water  = run_water_agent(ctx)
    urban  = run_urban_agent(ctx)
    carbon = run_carbon_agent(ctx, plant)
    soil   = run_soil_recommendations(ctx)
    energy = run_energy_agent(ctx)
    report = run_synthesis(ctx, pol, plant, water, urban, carbon, soil, energy, user_type)
    if polish and LLM.kind != "none":
        polished = LLM.generate(
            SYSTEM_PROMPTS["synthesis"] + f"\nUser type: {user_type}. Keep ALL numbers, species names and "
            "tables EXACTLY as given. Improve flow and tone only; do not add facts.",
            f"User query: {query}\n\nDraft report to refine:\n{report}", max_tokens=1800)
        if polished and len(polished) > 400:
            report = polished
    ask_ecogpt.last = {"ctx":ctx,"pol":pol,"plant":plant,"water":water,
                       "urban":urban,"carbon":carbon,"soil":soil,"energy":energy}
    return report

def ask_followup(question: str) -> str:
    """Free-form Q&A: hybrid RAG retrieval + last pipeline context + LLM."""
    chunks = retrieve(question, 4)
    ctx = getattr(ask_ecogpt, "last", {}).get("ctx")
    evidence = "\n\n".join(f"[{c['source']} | rel {c['relevance']}]\n{c['text']}" for c in chunks)
    if LLM.kind != "none":
        ans = LLM.generate(
            "You are EcoGPT, an environmental advisory assistant. Answer ONLY from the evidence and "
            "context provided. Cite sources inline. If the evidence is insufficient, say so explicitly.",
            f"Context: {json.dumps(ctx, default=str)[:1500] if ctx else 'none yet'}\n\n"
            f"Evidence:\n{evidence}\n\nQuestion: {question}")
        if ans: return ans
    # deterministic fallback: return best evidence verbatim
    return ("**Most relevant knowledge found:**\n\n" +
            "\n\n".join(f"**{c['source']}** (relevance {c['relevance']}):\n{c['text'][:600]}…"
                        for c in chunks[:2]) +
            "\n\n*(Connect Ollama or set HF_TOKEN for synthesized answers.)*")

# ---------- Google ADK wiring (used when google-adk is installed) ----------
ADK_READY = False
if CAPS["google.adk"]:
    try:
        from google.adk.agents import LlmAgent, SequentialAgent
        from google.adk.tools import FunctionTool
        from google.adk.runners import InMemoryRunner
        _model = f"ollama/{LLM.model}" if LLM.kind == "ollama" else "gemini-2.0-flash"
        _tools = {n: FunctionTool(f) for n, f in [
            ("load_sensor", load_sensor_data), ("aqi", compute_aqi),
            ("geocode", reverse_geocode), ("population", get_population_density),
            ("retrieve", retrieve), ("species", filter_species_by_climate),
            ("simulate", simulate_intervention)]}
        _agents = []
        for name, prompt, tool_keys in [
            ("data_ingestion_agent", SYSTEM_PROMPTS["data_ingestion"], ["load_sensor","aqi","geocode","population"]),
            ("rag_retrieval_agent",  SYSTEM_PROMPTS["rag_retrieval"],  ["retrieve"]),
            ("pollution_agent",      SYSTEM_PROMPTS["pollution"],      ["aqi","retrieve"]),
            ("plantation_agent",     SYSTEM_PROMPTS["plantation"],     ["species","retrieve"]),
            ("water_agent",          SYSTEM_PROMPTS["water"],          ["retrieve"]),
            ("urban_planning_agent", SYSTEM_PROMPTS["urban"],          ["population","retrieve"]),
            ("carbon_agent",         SYSTEM_PROMPTS["carbon"],         ["retrieve","simulate"]),
            ("synthesis_agent",      SYSTEM_PROMPTS["synthesis"],      [])]:
            _agents.append(LlmAgent(name=name, model=_model, instruction=prompt,
                                    tools=[_tools[k] for k in tool_keys]))
        adk_orchestrator = SequentialAgent(name="ecogpt_orchestrator", sub_agents=_agents)
        adk_runner = InMemoryRunner(agent=adk_orchestrator)
        ADK_READY = True
        print("Google ADK orchestrator wired ✓ (SequentialAgent, 8 sub-agents)")
    except Exception as e:
        print(f"ADK present but wiring failed ({e}) — built-in orchestrator in use")
else:
    print("google-adk not installed — built-in sequential orchestrator in use "
          "(identical agent prompts & tools; pip install google-adk to switch)")

async def ask_ecogpt_adk(location_query: str) -> str:
    """Run the pipeline through Google ADK (requires ADK + a live model)."""
    if not ADK_READY:
        return ask_ecogpt(location_query)
    session = await adk_runner.session_service.create_session(app_name="ecogpt", user_id="user")
    from google.genai import types as _t
    out = []
    async for ev in adk_runner.run_async(user_id="user", session_id=session.id,
            new_message=_t.Content(role="user", parts=[_t.Part(text=location_query)])):
        if ev.content and ev.content.parts:
            out += [p.text for p in ev.content.parts if getattr(p, "text", None)]
    return "\n".join(out)

print("Orchestrator ready ✓  → ask_ecogpt('Kolkata'), ask_followup('…')")


In [ ]:
# ============================================================
# Cell 10: Interactive Chat Interface
#   First message → full pipeline report for that location.
#   Follow-ups → RAG-grounded Q&A. Type a city or "lat, lon".
# ============================================================
if CAPS["ipywidgets"]:
    import ipywidgets as W
    from IPython.display import display, Markdown, clear_output

    inp = W.Text(placeholder="e.g. Kolkata  |  22.557, 88.494  |  How do I restore a pond?",
                 layout=W.Layout(width="70%"))
    user_type = W.Dropdown(options=["default","government","ngo","researcher"],
                           description="Audience:")
    show_dash = W.Checkbox(value=True, description="Auto-show dashboard after report")
    go = W.Button(description="Ask EcoGPT 🌿", button_style="success")
    out = W.Output()
    _first = {"done": False}

    def _on_go(_):
        q = inp.value.strip()
        if not q: return
        with out:
            clear_output()
            print("⏳ EcoGPT multi-agent pipeline running …")
            is_question = q.rstrip().endswith("?") or bool(
                re.match(r"(?i)^(how|what|which|why|where|when|who|can|could|should|tell|explain|compare|list)\b", q))
            loc = None if (is_question and _first["done"]) else parse_location(q)
            if loc is not None:
                md = ask_ecogpt(q, user_type=user_type.value)
                if not md.startswith("❓"): _first["done"] = True
            elif _first["done"]:
                md = ask_followup(q)
            else:
                md = ask_ecogpt(q, user_type=user_type.value)  # returns LOCATION_HELP if unresolvable
            clear_output(); display(Markdown(md))
            if loc is not None and not md.startswith("❓"):
                print(f"📍 Resolved location: {loc[2]} ({loc[0]:.4f}, {loc[1]:.4f})")
                # auto-render the satellite dashboard for this location
                if show_dash.value and "build_dashboard" in globals():
                    print("🛰  Building visualization dashboard …")
                    try:
                        build_dashboard(loc[2], lat=loc[0], lon=loc[1])
                    except Exception as e:
                        print(f"dashboard error: {e}")
                elif "build_dashboard" not in globals():
                    print("Run Cell 11 once, then dashboards auto-render here after each report.")
    go.on_click(_on_go)
    inp.on_submit(_on_go)
    display(W.VBox([W.HBox([inp, go]), W.HBox([user_type, show_dash]), out]))
    print("Tip: first ask a location (full report), then ask follow-ups "
          "(e.g. 'which species fight NO2?', 'how much water can a school roof harvest?')")
else:
    print("ipywidgets unavailable — call ask_ecogpt('Kolkata') / ask_followup('…') directly.")


In [ ]:
# ============================================================
# Cell 11: Visualization Dashboard — satellite-grounded
#   • Esri World Imagery satellite basemap (real imagery, no key)
#   • NASA GIBS MODIS NDVI overlay (real vegetation satellite data)
#   • REAL open/plantable land + water bodies from OpenStreetMap
#     (parks, meadows, brownfields, vacant lots) with per-plot
#     tree capacity at 1 tree / 25 m²
#   • Species-mix treemap + AQI gauge + CO2 projection
#   Works for ANY location: build_dashboard("Paris"), ("Nairobi"),
#   (lat=22.55, lon=88.49), build_dashboard("Bandipur, Nepal") …
# ============================================================
ZONE_COLORS = {"open": "#2ecc40", "water": "#0074d9"}

def build_dashboard(location="Kolkata", lat=None, lon=None, radius_m=3000):
    # ---- resolve location (same resolver as the chat pipeline) ----
    if lat is None or lon is None:
        loc = parse_location(str(location))
        if loc is None:
            print(LOCATION_HELP); return None
        lat, lon, label = loc
    else:
        label = str(location)
    ctx = run_data_ingestion(lat, lon, label=label)
    plant = run_plantation_agent(ctx); carbon = run_carbon_agent(ctx, plant)
    water = run_water_agent(ctx); energy = run_energy_agent(ctx)
    print(f"📍 {ctx['location']['city']}, {ctx['location']['country']} "
          f"({lat:.4f}, {lon:.4f}) | AQI {ctx['aqi']['score']} ({ctx['aqi']['category']})")
    print(f"📡 {ctx['data_source']} | ☀️ {ctx['solar_kwh_m2_day']} kWh/m²/day | "
          f"💨 {ctx['wind_ms']} m/s | 🏔 {energy['terrain']['slope_class']}")

    # ---- REAL plantable land from OpenStreetMap/Overpass ----
    spaces = get_open_spaces(lat, lon, radius_m)
    open_plots = [s for s in spaces if s["kind"] == "open"]
    water_bodies = [s for s in spaces if s["kind"] == "water"]
    total_open_ha = sum(s["area_m2"] for s in open_plots) / 10_000
    total_capacity = sum(s["tree_capacity"] for s in open_plots)
    if spaces:
        print(f"🛰️  OSM land survey ({radius_m/1000:.0f} km radius): "
              f"{len(open_plots)} open/plantable plots ({total_open_ha:.1f} ha, "
              f"capacity ≈{total_capacity:,} trees @25 m²/tree) | {len(water_bodies)} water bodies")
    else:
        print("🛰️  Overpass/OSM unreachable — showing illustrative zones (marked as such)")

    if CAPS["folium"]:
        from folium.plugins import HeatMap, MarkerCluster
        m = folium.Map(location=[lat, lon], zoom_start=14, tiles=None)
        folium.TileLayer("CartoDB positron", name="Street map").add_to(m)
        folium.TileLayer(
            tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
            attr="Esri World Imagery", name="🛰 Satellite imagery").add_to(m)
        try:  # NASA GIBS MODIS NDVI — real vegetation index, 8-day composite
            ndvi_date = (pd.Timestamp.now() - pd.Timedelta(days=12)).strftime("%Y-%m-%d")
            folium.TileLayer(
                tiles=("https://gibs.earthdata.nasa.gov/wmts/epsg3857/best/MODIS_Terra_NDVI_8Day/"
                       f"default/{ndvi_date}/GoogleMapsCompatible_Level9/{{z}}/{{y}}/{{x}}.png"),
                attr="NASA GIBS MODIS NDVI", name="🌱 Vegetation index (NDVI, satellite)",
                overlay=True, opacity=0.55, max_native_zoom=9).add_to(m)
        except Exception: pass

        # sensor heatmap (only where we actually have readings)
        local = load_sensor_data(lat, lon, 0.2)
        if len(local) > 30:
            pts = local.dropna(subset=["LAT","LON","CO"]).sample(min(1500, len(local)), random_state=1)
            HeatMap(pts[["LAT","LON","CO"]].values.tolist(), radius=18, blur=22,
                    name="🔥 CO pollution heat (sensors)", show=False).add_to(m)

        if spaces:
            fg_open = folium.FeatureGroup(name=f"🌳 Plantable land ({len(open_plots)} plots, OSM)")
            fg_water = folium.FeatureGroup(name=f"💧 Water bodies ({len(water_bodies)}, OSM)")
            cluster = MarkerCluster(name="📌 Proposed planting sites")
            top_species = [s["common"] for s in plant["species"][:4]]
            for s in open_plots:
                folium.Polygon(s["coords"], color=ZONE_COLORS["open"], weight=2,
                    fill=True, fill_opacity=0.35,
                    tooltip=(f"🌳 {s['name']} — {s['area_m2']/10000:.2f} ha | plantable capacity "
                             f"≈{s['tree_capacity']:,} trees | suggest: {', '.join(top_species)}")
                    ).add_to(fg_open)
                cy = sum(c[0] for c in s["coords"])/len(s["coords"])
                cx = sum(c[1] for c in s["coords"])/len(s["coords"])
                folium.Marker([cy, cx], icon=folium.Icon(color="green", icon="leaf"),
                    popup=folium.Popup(
                        f"<b>{s['name']}</b><br>{s['area_m2']:,} m² open land<br>"
                        f"≈<b>{s['tree_capacity']:,} trees</b> @25 m² spacing<br>"
                        f"Mix: {', '.join(top_species)}<br>"
                        f"Layout: {plant['spatial_arrangement'][:80]}…", max_width=260)).add_to(cluster)
            for s in water_bodies:
                folium.Polygon(s["coords"], color=ZONE_COLORS["water"], weight=2,
                    fill=True, fill_opacity=0.3,
                    tooltip=(f"💧 {s['name']} — {s['area_m2']/10000:.2f} ha | riparian buffer: "
                             "Arjun/Jamun/Vetiver; reed beds at inlets")).add_to(fg_water)
            fg_open.add_to(m); fg_water.add_to(m); cluster.add_to(m)
        else:  # offline fallback — clearly labelled illustrative rings
            for name, dy, dx in [("Illustrative: micro-forest", .010,.010),
                                 ("Illustrative: avenue corridor", -.012,.008),
                                 ("Illustrative: riparian buffer", .008,-.013)]:
                folium.Circle([lat+dy, lon+dx], radius=300, color="green", fill=True,
                              fill_opacity=0.2, tooltip=name + " (no OSM data — placeholder)").add_to(m)

        folium.Marker([lat, lon],
            tooltip=f"{ctx['location']['city']} — AQI {ctx['aqi']['score']} ({ctx['aqi']['category']})",
            icon=folium.Icon(color="red" if ctx["aqi"]["score"]>150 else "orange"
                             if ctx["aqi"]["score"]>100 else "green", icon="info-sign")).add_to(m)
        folium.LayerControl(collapsed=False).add_to(m)
        display(m)

    # ---- species-mix treemap (capacity allocated across recommended mix) ----
    cap = total_capacity if total_capacity else 10000
    sp = plant["species"][:10]
    weights = np.array([s["co2_kg_yr"] for s in sp], dtype=float); weights /= weights.sum()
    alloc = (weights * cap).astype(int)
    if CAPS["plotly"]:
        import plotly.express as px
        tm = px.treemap(
            pd.DataFrame({"species": [f"{s['common']}<br>({s['scientific']})" for s in sp],
                          "trees": alloc,
                          "co2": [int(a*s["co2_kg_yr"]/1000) for a, s in zip(alloc, sp)],
                          "layer": [s["layer"] for s in sp]}),
            path=["layer", "species"], values="trees", color="co2",
            color_continuous_scale="Greens",
            title=f"Proposed planting mix for {ctx['location']['city']} — "
                  f"{cap:,} trees across {total_open_ha:.1f} ha of identified open land"
                  if total_capacity else
                  f"Proposed planting mix for {ctx['location']['city']} (10,000-tree plan)")
        tm.update_layout(margin=dict(t=50,l=10,r=10,b=10), height=420)
        tm.show()
    else:
        fig, ax = plt.subplots(figsize=(9, 3.5))
        ax.barh([s["common"] for s in sp][::-1], alloc[::-1], color="forestgreen")
        ax.set_title(f"Planting mix, {ctx['location']['city']} (≈{cap:,} trees)")
        plt.tight_layout(); plt.show()

    # ================= PRESENTATION DASHBOARD (8 panels, all live data) =================
    plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False,
                         "axes.spines.right": False, "axes.titlesize": 11,
                         "axes.titleweight": "bold", "figure.facecolor": "white"})
    GRN, RED, BLU, ORG, GRY = "#2e7d32", "#c62828", "#1565c0", "#ef6c00", "#9e9e9e"
    fig, ax = plt.subplots(4, 2, figsize=(15.5, 16))
    fig.suptitle(f"EcoGPT Impact Dashboard — {ctx['location']['city']}, {ctx['location']['country']}   "
                 f"|   {ctx['data_source'].split('(')[0].strip()}", fontsize=14, fontweight="bold", y=0.995)
    s = ctx["stats"]; gm = lambda c: s.get(c, {}).get("mean", float("nan"))

    # 1 — AQI gauge
    a = ax[0,0]; left = 0
    for ceil_, col in [(50,"#00e400"),(100,"#ffff00"),(150,"#ff7e00"),
                       (200,"#ff0000"),(300,"#8f3f97"),(400,"#7e0023")]:
        a.barh(0, ceil_-left, left=left, color=col, height=0.45); left = ceil_
    a.axvline(min(ctx["aqi"]["score"], 400), color="black", lw=4)
    a.annotate(f"{ctx['aqi']['score']}", (min(ctx['aqi']['score'],395), 0.32),
               fontsize=15, fontweight="bold", ha="center")
    a.set(xlim=(0,400), yticks=[], title=f"1 · Air Quality Index — {ctx['aqi']['category']} "
          f"(dominant: {ctx['aqi']['dominant']})")

    # 2 — Pollutants vs WHO 24-h guideline values
    a = ax[0,1]
    pm25, pm10 = gm("RAWPM"), gm("DD")
    no2_ug = gm("NO2")*1000*1.88            # ppm → µg/m³
    co_mg = gm("CO")*1.145                  # ppm → mg/m³
    names = ["PM2.5\n(µg/m³)","PM10\n(µg/m³)","NO2\n(µg/m³)","CO\n(mg/m³)"]
    vals = [pm25, pm10, no2_ug, co_mg]; who = [15, 45, 25, 4]
    ratio = [v/w if v==v else 0 for v, w in zip(vals, who)]
    bars = a.bar(names, ratio, color=[RED if r > 1 else GRN for r in ratio], width=0.55)
    a.axhline(1, color="black", ls="--", lw=1.5)
    a.text(3.45, 1.04, "WHO limit", fontsize=8)
    for b, v, r in zip(bars, vals, ratio):
        if v == v:
            a.annotate(f"{v:.0f}\n({r:.1f}×)", (b.get_x()+b.get_width()/2, r),
                       ha="center", va="bottom", fontsize=8)
    a.set(title="2 · Current pollution vs WHO guidelines (× limit)", ylabel="× WHO 24-h limit")

    # 3 — CO2 sequestration projection
    a = ax[1,0]
    yrs = [1,5,10,25]; vv = [carbon["cumulative_tonnes_co2e"][f"year_{y}"] for y in yrs]
    a.plot(yrs, vv, "o-", color=GRN, lw=2.5, ms=7)
    a.fill_between(yrs, vv, alpha=0.18, color=GRN)
    for x, y in zip(yrs, vv):
        a.annotate(f"{y:,.0f} t", (x, y), textcoords="offset points", xytext=(0,8),
                   ha="center", fontsize=8.5)
    a.set(title=f"3 · Carbon captured by {carbon['trees_modelled']:,} trees "
          f"(≈{carbon['cars_equivalent_at_maturity']:,} cars/yr at maturity)",
          xlabel="years after planting", ylabel="cumulative t CO2e")

    # 4 — Green space: measured open land vs WHO need
    a = ax[1,1]
    area_km2 = math.pi*(radius_m/1000)**2
    pop_in_area = ctx["population_density"]*area_km2
    current_m2pc = (total_open_ha*10_000/pop_in_area) if (spaces and pop_in_area>0) else float("nan")
    bars_v = [current_m2pc if current_m2pc==current_m2pc else 0, 9]
    a.bar(["Open/green land\nfound (OSM survey)","WHO minimum\ntarget"], bars_v,
          color=[ORG if bars_v[0] < 9 else GRN, GRY], width=0.5)
    for i, v in enumerate(bars_v):
        a.annotate(f"{v:.1f} m²/person", (i, v), ha="center", va="bottom", fontsize=10, fontweight="bold")
    a.set(title=f"4 · Green space per capita within {radius_m/1000:.0f} km"
          + ("" if spaces else " (offline — no survey)"), ylabel="m² per person")

    # 5 — Renewable resource quality at this site
    a = ax[2,0]
    a.bar(["Solar\n(kWh/m²/day)","Wind\n(m/s)"], [energy["solar_kwh_m2_day"], energy["wind_ms"]],
          color=[ORG, BLU], width=0.45)
    a.axhline(3.5, xmin=0.05, xmax=0.45, color=ORG, ls="--"); a.text(-0.38, 3.6, "solar viable ≥3.5", fontsize=8, color=ORG)
    a.axhline(4.0, xmin=0.55, xmax=0.95, color=BLU, ls="--"); a.text(0.62, 4.1, "wind viable ≥4", fontsize=8, color=BLU)
    feas = sum(1 for i in energy["interventions"] if i["feasible"].startswith("✅"))
    a.set(title=f"5 · Renewable resources here — terrain {energy['terrain']['slope_class']} "
          f"({feas}/{len(energy['interventions'])} interventions feasible)")

    # 6 — What-if: AQI after plantation scenarios
    a = ax[2,1]
    base_n = {"Rural":50000,"Peri-urban":25000,"Urban":15000,"Dense Urban":10000}[ctx["density_class"]]
    scen = [(base_n,5),(base_n*3,10),(base_n*6,25)]
    sims = [simulate_intervention(ctx["aqi"]["score"], n, n/plant["trees_per_hectare"], y) for n,y in scen]
    lbl = [f"{n//1000}k trees\n{y} yrs" for n,y in scen]
    a.bar(lbl, [ctx["aqi"]["score"]]*3, color=GRY, width=0.55, label="today")
    a.bar(lbl, [s_["projected_aqi"] for s_ in sims], color=GRN, width=0.55, label="projected")
    for i, s_ in enumerate(sims):
        a.annotate(f"−{s_['aqi_reduction']}", (i, s_["projected_aqi"]),
                   ha="center", va="bottom", fontsize=9, fontweight="bold", color="white")
    a.legend(fontsize=8); a.set(title="6 · What-if: AQI impact of plantation scenarios")

    # 7 — Water: rooftop harvest vs household demand
    a = ax[3,0]
    rwh = water["rainwater_harvesting"]["per_100m2_roof_litres_yr"]
    demand = 5*135*365
    a.bar(["Harvestable\n(100 m² roof)","5-person household\ndemand (135 LPCD)"],
          [rwh/1000, demand/1000], color=[BLU, GRY], width=0.5)
    for i, v in enumerate([rwh/1000, demand/1000]):
        a.annotate(f"{v:,.0f} kL/yr", (i, v), ha="center", va="bottom", fontsize=10, fontweight="bold")
    a.set(title=f"7 · Rainwater harvesting potential ({water['annual_rainfall_mm']} mm rain/yr → "
          f"covers {water['rainwater_harvesting']['household_demand_coverage_pct']}%)", ylabel="kilolitres/yr")

    # 8 — Heat: actual vs feels-like
    a = ax[3,1]
    delta = ctx["heat_index_avg"] - ctx["temperature_avg"]
    a.bar(["Air temperature","Feels like\n(heat index)"],
          [ctx["temperature_avg"], ctx["heat_index_avg"]],
          color=[GRY, RED if delta > 3 else ORG], width=0.5)
    for i, v in enumerate([ctx["temperature_avg"], ctx["heat_index_avg"]]):
        a.annotate(f"{v:.1f}°C", (i, v), ha="center", va="bottom", fontsize=10, fontweight="bold")
    a.set(title=f"8 · Heat island: +{delta:.1f}°C feels-like "
          f"({'⚠ mitigation needed' if delta > 3 else 'acceptable'})", ylabel="°C")

    plt.tight_layout(rect=[0, 0, 1, 0.98]); plt.show()
    print("How to read: 1-2 current state · 3-6 impact of recommendations · "
          "7-8 water & heat context. All values computed for THIS location.")
    return {"ctx": ctx, "open_plots": open_plots, "water_bodies": water_bodies,
            "plantable_ha": round(total_open_ha,1), "tree_capacity": total_capacity,
            "energy": energy}

# Dashboard follows whatever location you last asked about in the chat (Cell 10);
# falls back to the sensor reference site only if no query has been made yet.
_last = getattr(ask_ecogpt, "last", None)
if _last:
    L = _last["ctx"]["location"]
    dash = build_dashboard(L["city"], lat=L["lat"], lon=L["lon"])
else:
    dash = build_dashboard("Kolkata")   # reference sensor site — try build_dashboard("Paris"), ("Nairobi", radius_m=5000)


In [ ]:
# ============================================================
# Cell 12: Demo Scenarios
#   1) Kolkata dense urban  2) Sundarbans rural/riverine
#   3) Global check — London (temperate, different species set)
# ============================================================
from IPython.display import Markdown, display

print("="*70); print("DEMO 1 — Kolkata (dense urban, real reference-sensor region)"); print("="*70)
display(Markdown(ask_ecogpt(
    "Location: Kolkata, West Bengal (lat: 22.557, lon: 88.494). "
    "How can we improve the environment here? Focus on water bodies, tree planting, "
    "and pollution reduction given the high population density.", polish=False)))

print("="*70); print("DEMO 2 — Sundarbans (rural riverbank restoration)"); print("="*70)
display(Markdown(ask_ecogpt("Sundarbans riverbank restoration, lat: 21.9497, lon: 88.9468",
                            user_type="ngo", polish=False)))

print("="*70); print("DEMO 3 — London (global generalisation, temperate species)"); print("="*70)
display(Markdown(ask_ecogpt("London", user_type="government", polish=False)))


In [ ]:
# ============================================================
# Cell 13: What-If Simulation Playground
#   Tune the sliders / arguments and compare intervention scenarios
# ============================================================
ctx = getattr(ask_ecogpt, "last", {}).get("ctx") or run_data_ingestion(22.557, 88.494)
baseline = ctx["aqi"]["score"]

scenarios = {
    "Conservative (5k trees / 3 yr)":   simulate_intervention(baseline, 5000, 50, 3, 50_000),
    "Municipal plan (25k / 10 yr)":     simulate_intervention(baseline, 25000, 250, 10, 400_000),
    "Moonshot (100k / 25 yr)":          simulate_intervention(baseline, 100000, 1000, 25, 2_000_000),
}
cmp_df = pd.DataFrame(scenarios).T
display(cmp_df)

fig, ax = plt.subplots(figsize=(9, 3.5))
names = list(scenarios)
ax.bar(names, [baseline]*3, color="#cccccc", label="baseline AQI")
ax.bar(names, [s["projected_aqi"] for s in scenarios.values()], color="forestgreen",
       label="projected AQI")
for i, s in enumerate(scenarios.values()):
    ax.text(i, s["projected_aqi"]+3, f"−{s['aqi_reduction']}", ha="center", fontsize=9)
ax.set_title(f"What-if: AQI impact of plantation scenarios ({ctx['location']['city']})")
ax.legend(); plt.xticks(fontsize=8); plt.tight_layout(); plt.show()

if CAPS["ipywidgets"]:
    import ipywidgets as W
    @W.interact(trees=W.IntSlider(10000, 1000, 200000, 1000),
                years=W.IntSlider(10, 1, 30),
                water_m3=W.IntSlider(100000, 0, 5_000_000, 50000))
    def _sim(trees, years, water_m3):
        s = simulate_intervention(baseline, trees, trees/150, years, water_m3)
        print(json.dumps(s, indent=2))


---
## ⚡ AMD ROCm acceleration notes

```bash
# Ollama on AMD GPU (ROCm) — Linux
curl -fsSL https://ollama.com/install.sh | sh
ollama pull mistral:7b-instruct        # or llama3.1:8b
# Verify GPU offload: ollama ps  →  "100% GPU"
```

```python
import torch; print(torch.cuda.is_available())   # True on ROCm builds of PyTorch
# 4-bit quantized alternative for vLLM/transformers on ROCm:
# model_id = "TheBloke/Mistral-7B-Instruct-v0.3-AWQ"
```

## 🧭 How the design maps to the hackathon rubric

| Criterion | Where in this notebook |
|---|---|
| Innovation | 8-agent pipeline + hybrid RAG (dense∪BM25, RRF fusion) over live IoT streams (Cells 5–9) |
| Technical depth | EPA AQI interpolation, source apportionment, IPCC-style carbon ramp w/ mortality, Köppen-aware species DB (Cells 6–8) |
| Real-world impact | Named native species, quantified L/yr & tCO2e, density-scaled plans, regulatory grounding (Cells 8–9) |
| AMD relevance | Ollama/ROCm-served Mistral-7B; AWQ 4-bit path documented above |
| Open source | Mistral/Llama, ChromaDB, bge-small, sentence-transformers, Google ADK |
| Presentation | Folium heatmaps + zones, AQI gauge, what-if simulator, live chat widget (Cells 10–13) |

**Degradation ladder (works for anyone, anywhere):** ADK + Ollama GPU → built-in orchestrator + Ollama → HF Inference API → fully deterministic engine with RAG (no LLM, no network). Every number in fallback mode comes from the same scientific cores — only narrative polish is lost.
